# Sector Gamma Dashboard

米国 S&P 500 セクター ETF について、価格モメンタム、オプション需給、Black-Scholes Gamma から計算した GEX 指標を使い、セクター配分の Overweight / Underweight 候補を可視化する Notebook。

この Notebook は `codex_sector_gamma_dashboard_instructions.md` に基づき、読み込み、計算、可視化、保存、検証を上から順に実行できる構成にしている。


## 1. Overview

目的:

- 11 セクター ETF と SPY の価格データを yFinance から取得する。
- 各セクター ETF の現在のオプションチェーンを取得する。
- yFinance に Gamma がないため、`impliedVolatility` から Black-Scholes Gamma を計算する。
- Call Wall、Put Wall、Gamma Flip、Near-Spot GEX、Short-Dated GEX を算出する。
- Momentum / Gamma / Option Demand / Risk Control から総合スコアを作り、ベンチマーク対比の提案ウェイトを作成する。
- 直前の提案配分を翌週の配当調整済み終値で評価し、Benchmark対比の寄与度を検証する。
- Plotly による Dashboard と HTML / CSV / parquet 出力を作成する。

注意:

本ダッシュボードのGEXは、yFinanceから取得した現在のオプションチェーン、Open Interest、Implied Volatilityを用いてBlack-Scholes Gammaを自前計算した近似値である。

Signed GEXは、callを正、putを負とする簡易プロキシであり、実際のディーラーのロング/ショートGammaを直接表すものではない。

yFinanceのデータは研究・教育目的向けであり、オプションチェーン、IV、OIの更新タイミングや品質には制約がある。実運用や厳密なバックテストでは、ThetaData、ORATS、Cboe、OptionMetrics等の正式データソースへの差し替えを検討すること。


## 2. Config


In [1]:
from __future__ import annotations

import datetime as dt
import importlib.util
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm
import yfinance as yf

REQUIRED_MODULES = [
    "pandas",
    "numpy",
    "scipy",
    "yfinance",
    "matplotlib",
    "plotly",
    "ipywidgets",
    "pyarrow",
]

missing_modules = [m for m in REQUIRED_MODULES if importlib.util.find_spec(m) is None]
if missing_modules:
    raise ImportError(
        "Missing required modules: "
        + ", ".join(missing_modules)
        + ". Add them under /Users/kencharoff/workspace/envs/base with uv add before running this notebook."
    )

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import HTML, display

warnings.filterwarnings("once")
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 160)
pio.templates.default = "plotly_white"

PROJECT_ROOT = Path.cwd()
AS_OF = pd.Timestamp.now(tz="Asia/Tokyo").strftime("%Y%m%d")

SECTOR_ETFS = [
    "XLK",
    "XLY",
    "XLP",
    "XLE",
    "XLF",
    "XLV",
    "XLI",
    "XLC",
    "XLB",
    "XLRE",
    "XLU",
]

SECTOR_NAMES = {
    "XLK": "Information Technology",
    "XLY": "Consumer Discretionary",
    "XLP": "Consumer Staples",
    "XLE": "Energy",
    "XLF": "Financials",
    "XLV": "Health Care",
    "XLI": "Industrials",
    "XLC": "Communication Services",
    "XLB": "Materials",
    "XLRE": "Real Estate",
    "XLU": "Utilities",
}

BENCHMARK_WEIGHTS = {
    "XLK": 0.329,
    "XLF": 0.126,
    "XLC": 0.103,
    "XLY": 0.099,
    "XLV": 0.095,
    "XLI": 0.090,
    "XLP": 0.053,
    "XLE": 0.040,
    "XLU": 0.025,
    "XLB": 0.021,
    "XLRE": 0.020,
}

PRICE_PERIOD = "1y"
MAX_EXPIRIES = None
RISK_FREE_RATE = 0.045
DIVIDEND_YIELD = 0.0
IV_MAX = 5.0
IV_EXPECTED_MOVE_DAYS = 7
IV_EXPECTED_MOVE_Z = 1.0

RAW_OPTIONS_DIR = PROJECT_ROOT / "data" / "raw" / "options" / AS_OF
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / AS_OF
DASHBOARD_DIR = PROJECT_ROOT / "outputs" / "dashboard"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"

print(f"Project root: {PROJECT_ROOT}")
print(f"As of: {AS_OF}")


Project root: /Users/kencharoff/workspace/projects/sector-gamma
As of: 20260721


## 3. Price Data


In [2]:
def _pick_close_frame(downloaded: pd.DataFrame) -> pd.DataFrame:
    if downloaded.empty:
        return pd.DataFrame()
    if isinstance(downloaded.columns, pd.MultiIndex):
        first_level = list(downloaded.columns.get_level_values(0).unique())
        if "Close" in first_level:
            close = downloaded["Close"]
        elif "Adj Close" in first_level:
            close = downloaded["Adj Close"]
        else:
            raise ValueError("Downloaded price data does not include Close or Adj Close.")
        if isinstance(close, pd.Series):
            close = close.to_frame()
        return close
    if "Close" in downloaded.columns:
        return downloaded[["Close"]]
    if "Adj Close" in downloaded.columns:
        return downloaded[["Adj Close"]].rename(columns={"Adj Close": "Close"})
    raise ValueError("Downloaded price data does not include Close or Adj Close.")


def get_price_data(symbols, period="1y") -> pd.DataFrame:
    raw = yf.download(
        tickers=list(symbols),
        period=period,
        auto_adjust=True,
        progress=False,
        group_by="column",
        threads=True,
    )
    close = _pick_close_frame(raw)
    if close.empty:
        return pd.DataFrame(columns=["date", "symbol", "close"])

    if len(symbols) == 1 and list(close.columns) == ["Close"]:
        close = close.rename(columns={"Close": symbols[0]})

    close.index = pd.to_datetime(close.index).tz_localize(None)
    close = close.sort_index()
    price_df = (
        close.reset_index()
        .rename(columns={close.index.name or "index": "date"})
        .melt(id_vars="date", var_name="symbol", value_name="close")
    )
    price_df["close"] = pd.to_numeric(price_df["close"], errors="coerce")
    price_df = price_df.dropna(subset=["close"])
    price_df["symbol"] = price_df["symbol"].astype(str)
    return price_df.sort_values(["symbol", "date"]).reset_index(drop=True)


def _safe_last(series: pd.Series):
    values = series.dropna()
    if values.empty:
        return np.nan
    return values.iloc[-1]


def compute_price_signals(price_df: pd.DataFrame) -> pd.DataFrame:
    columns = [
        "symbol",
        "price",
        "ret20",
        "ret60",
        "rel_ret20_vs_spy",
        "rel_ret60_vs_spy",
        "ma20",
        "ma50",
        "dd_60",
        "above_20ma",
        "above_50ma",
        "drawdown_gt_7pct",
        "momentum_score",
    ]
    if price_df.empty:
        return pd.DataFrame(columns=columns)

    prices = price_df.pivot(index="date", columns="symbol", values="close").sort_index()
    spy = prices["SPY"] if "SPY" in prices.columns else pd.Series(index=prices.index, dtype=float)
    spy_ret20 = _safe_last(spy / spy.shift(20) - 1)
    spy_ret60 = _safe_last(spy / spy.shift(60) - 1)

    records = []
    for symbol in SECTOR_ETFS:
        if symbol not in prices.columns:
            continue
        s = prices[symbol].dropna()
        if s.empty:
            continue

        price = _safe_last(s)
        ret20 = _safe_last(s / s.shift(20) - 1)
        ret60 = _safe_last(s / s.shift(60) - 1)
        ma20 = _safe_last(s.rolling(20, min_periods=20).mean())
        ma50 = _safe_last(s.rolling(50, min_periods=50).mean())
        high_60 = _safe_last(s.rolling(60, min_periods=20).max())
        dd_60 = price / high_60 - 1 if pd.notna(price) and pd.notna(high_60) and high_60 != 0 else np.nan
        above_20ma = bool(pd.notna(price) and pd.notna(ma20) and price > ma20)
        above_50ma = bool(pd.notna(price) and pd.notna(ma50) and price > ma50)
        drawdown_gt_7pct = bool(pd.notna(dd_60) and dd_60 <= -0.07)
        rel_ret20 = ret20 - spy_ret20 if pd.notna(ret20) and pd.notna(spy_ret20) else np.nan
        rel_ret60 = ret60 - spy_ret60 if pd.notna(ret60) and pd.notna(spy_ret60) else np.nan

        momentum_score = 0.0
        momentum_score += 0.50 if pd.notna(ret20) and ret20 > 0 else -0.50
        momentum_score += 0.50 if pd.notna(ret60) and ret60 > 0 else -0.50
        momentum_score += 0.50 if pd.notna(rel_ret20) and rel_ret20 > 0 else -0.25
        momentum_score += 0.50 if pd.notna(rel_ret60) and rel_ret60 > 0 else -0.25
        momentum_score += 0.25 if above_20ma else -0.25
        momentum_score += 0.25 if above_50ma else -0.25
        if drawdown_gt_7pct:
            momentum_score -= 0.75

        records.append(
            {
                "symbol": symbol,
                "price": price,
                "ret20": ret20,
                "ret60": ret60,
                "rel_ret20_vs_spy": rel_ret20,
                "rel_ret60_vs_spy": rel_ret60,
                "ma20": ma20,
                "ma50": ma50,
                "dd_60": dd_60,
                "above_20ma": above_20ma,
                "above_50ma": above_50ma,
                "drawdown_gt_7pct": drawdown_gt_7pct,
                "momentum_score": momentum_score,
            }
        )

    return pd.DataFrame(records, columns=columns)


WEEKLY_ATTRIBUTION_COLUMNS = [
    "symbol",
    "sector_name",
    "prior_decision",
    "prior_snapshot",
    "start_date",
    "end_date",
    "start_close",
    "end_close",
    "realized_return",
    "benchmark_weight",
    "active_weight",
    "proposed_weight",
    "benchmark_contribution",
    "active_contribution",
    "proposed_contribution",
    "prior_total_score",
    "prior_iv_adjusted_score",
    "prior_overweight_allowed",
]


def find_previous_processed_snapshot(processed_root: Path, as_of: str | pd.Timestamp = AS_OF) -> Path | None:
    current_date = pd.Timestamp(as_of).normalize()
    candidates = []
    if processed_root.exists():
        for path in processed_root.iterdir():
            if not path.is_dir() or not path.name.isdigit() or len(path.name) != 8:
                continue
            snapshot_date = pd.to_datetime(path.name, format="%Y%m%d", errors="coerce")
            if pd.notna(snapshot_date) and snapshot_date < current_date and (path / "proposed_portfolio.parquet").exists():
                candidates.append((snapshot_date, path))
    return max(candidates, key=lambda item: item[0])[1] if candidates else None


def _forecast_week_bounds(snapshot_date: pd.Timestamp) -> tuple[pd.Timestamp, pd.Timestamp]:
    snapshot_date = pd.Timestamp(snapshot_date).normalize()
    days_to_monday = (7 - snapshot_date.weekday()) % 7
    week_start = snapshot_date + pd.Timedelta(days=days_to_monday)
    return week_start, week_start + pd.Timedelta(days=4)


def compute_previous_week_attribution(
    price_df: pd.DataFrame,
    processed_root: Path = PROJECT_ROOT / "data" / "processed",
    as_of: str | pd.Timestamp = AS_OF,
) -> tuple[pd.DataFrame, dict]:
    empty = pd.DataFrame(columns=WEEKLY_ATTRIBUTION_COLUMNS)
    previous_dir = find_previous_processed_snapshot(processed_root, as_of=as_of)
    if previous_dir is None:
        warnings.warn("Previous processed snapshot was not found; weekly attribution is unavailable.")
        return empty, {}
    if price_df is None or price_df.empty:
        warnings.warn("Price data is empty; weekly attribution is unavailable.")
        return empty, {}

    snapshot_date = pd.to_datetime(previous_dir.name, format="%Y%m%d")
    week_start, week_end = _forecast_week_bounds(snapshot_date)
    prices = price_df.pivot(index="date", columns="symbol", values="close").sort_index()
    prices.index = pd.to_datetime(prices.index).tz_localize(None)
    start_candidates = prices.index[prices.index < week_start]
    end_candidates = prices.index[prices.index <= week_end]
    if len(start_candidates) == 0 or len(end_candidates) == 0:
        warnings.warn("Completed trading dates for weekly attribution were not found.")
        return empty, {}

    start_date = start_candidates.max()
    end_date = end_candidates.max()
    if end_date <= start_date:
        warnings.warn("Weekly attribution end date is not after the start date.")
        return empty, {}

    prior = pd.read_parquet(previous_dir / "proposed_portfolio.parquet")
    prior_cols = [
        "symbol", "sector_name", "benchmark_weight", "active_weight", "proposed_weight",
        "total_score", "iv_adjusted_score", "overweight_allowed",
    ]
    prior = prior[[c for c in prior_cols if c in prior.columns]].copy()
    prior = prior.rename(columns={
        "total_score": "prior_total_score",
        "iv_adjusted_score": "prior_iv_adjusted_score",
        "overweight_allowed": "prior_overweight_allowed",
    })
    prior = prior[prior["symbol"].isin(SECTOR_ETFS)].drop_duplicates("symbol")

    close_records = []
    for symbol in SECTOR_ETFS:
        start_close = prices.at[start_date, symbol] if symbol in prices.columns else np.nan
        end_close = prices.at[end_date, symbol] if symbol in prices.columns else np.nan
        realized_return = end_close / start_close - 1 if pd.notna(start_close) and pd.notna(end_close) and start_close != 0 else np.nan
        close_records.append({
            "symbol": symbol,
            "prior_snapshot": previous_dir.name,
            "start_date": start_date,
            "end_date": end_date,
            "start_close": start_close,
            "end_close": end_close,
            "realized_return": realized_return,
        })

    attribution = prior.merge(pd.DataFrame(close_records), on="symbol", how="right")
    if "sector_name" not in attribution:
        attribution["sector_name"] = attribution["symbol"].map(SECTOR_NAMES)
    else:
        attribution["sector_name"] = attribution["sector_name"].fillna(attribution["symbol"].map(SECTOR_NAMES))
    attribution["prior_decision"] = np.select(
        [attribution["active_weight"] > 1e-12, attribution["active_weight"] < -1e-12],
        ["Overweight", "Underweight"],
        default="Neutral",
    )
    for prefix in ["benchmark", "active", "proposed"]:
        attribution[f"{prefix}_contribution"] = attribution[f"{prefix}_weight"] * attribution["realized_return"]

    attribution = attribution.reindex(columns=WEEKLY_ATTRIBUTION_COLUMNS)
    complete = (
        set(attribution["symbol"]) == set(SECTOR_ETFS)
        and attribution[["realized_return", "benchmark_weight", "active_weight", "proposed_weight"]].notna().all().all()
    )
    summary = {
        "prior_snapshot": previous_dir.name,
        "start_date": start_date,
        "end_date": end_date,
        "benchmark_return": attribution["benchmark_contribution"].sum() if complete else np.nan,
        "proposed_return": attribution["proposed_contribution"].sum() if complete else np.nan,
        "active_return": attribution["active_contribution"].sum() if complete else np.nan,
        "complete": bool(complete),
    }
    return attribution.sort_values("active_contribution", ascending=False).reset_index(drop=True), summary


## 4. Option Chain Collection


In [3]:
def _latest_spot_from_yfinance(symbol: str) -> float:
    hist = yf.Ticker(symbol).history(period="5d", auto_adjust=True)
    if hist.empty or "Close" not in hist:
        return np.nan
    return float(hist["Close"].dropna().iloc[-1])


def _normalize_option_side(df: pd.DataFrame, symbol: str, option_type: str, expiration: str, spot: float) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()

    out = df.copy()
    out["symbol"] = symbol
    out["option_type"] = option_type
    out["expiration"] = pd.to_datetime(expiration).date().isoformat()
    out["spot"] = spot

    today = pd.Timestamp.now(tz="America/New_York").date()
    expiry_date = pd.to_datetime(expiration).date()
    dte = (expiry_date - today).days
    out["dte"] = dte
    out["T"] = dte / 365.0

    numeric_cols = [
        "strike",
        "lastPrice",
        "bid",
        "ask",
        "change",
        "percentChange",
        "volume",
        "openInterest",
        "impliedVolatility",
    ]
    for col in numeric_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
        else:
            out[col] = np.nan

    out["volume"] = out["volume"].fillna(0)
    out["openInterest"] = out["openInterest"].fillna(0)

    required_cols = [
        "symbol",
        "option_type",
        "expiration",
        "contractSymbol",
        "lastTradeDate",
        "strike",
        "lastPrice",
        "bid",
        "ask",
        "change",
        "percentChange",
        "volume",
        "openInterest",
        "impliedVolatility",
        "inTheMoney",
        "contractSize",
        "currency",
        "spot",
        "dte",
        "T",
    ]
    for col in required_cols:
        if col not in out.columns:
            out[col] = np.nan

    out = out[required_cols]
    out = out[out["T"] > 0]
    out = out[out["impliedVolatility"].between(0, IV_MAX, inclusive="neither")]
    out = out[out["strike"] > 0]
    out = out[pd.notna(out["spot"]) & (out["spot"] > 0)]
    return out.reset_index(drop=True)


def get_yfinance_option_chain(symbol, max_expiries=None) -> pd.DataFrame:
    empty_cols = [
        "symbol",
        "option_type",
        "expiration",
        "contractSymbol",
        "lastTradeDate",
        "strike",
        "lastPrice",
        "bid",
        "ask",
        "change",
        "percentChange",
        "volume",
        "openInterest",
        "impliedVolatility",
        "inTheMoney",
        "contractSize",
        "currency",
        "spot",
        "dte",
        "T",
    ]
    try:
        ticker = yf.Ticker(symbol)
        spot = _latest_spot_from_yfinance(symbol)
        expiries = list(ticker.options or [])
        if max_expiries is not None:
            expiries = expiries[: int(max_expiries)]
        frames = []
        for expiration in expiries:
            try:
                chain = ticker.option_chain(expiration)
                frames.append(_normalize_option_side(chain.calls, symbol, "call", expiration, spot))
                frames.append(_normalize_option_side(chain.puts, symbol, "put", expiration, spot))
            except Exception as exc:
                warnings.warn(f"{symbol} {expiration}: option chain fetch failed: {exc}")
        if not frames:
            return pd.DataFrame(columns=empty_cols)
        out = pd.concat(frames, ignore_index=True)
        return out[empty_cols].sort_values(["expiration", "option_type", "strike"]).reset_index(drop=True)
    except Exception as exc:
        warnings.warn(f"{symbol}: option chain fetch failed: {exc}")
        return pd.DataFrame(columns=empty_cols)


def fetch_all_option_chains(symbols, max_expiries=None) -> dict[str, pd.DataFrame]:
    chains = {}
    for symbol in symbols:
        print(f"Fetching option chain: {symbol}")
        chains[symbol] = get_yfinance_option_chain(symbol, max_expiries=max_expiries)
        print(f"  rows: {len(chains[symbol]):,}")
    return chains


## 5. Black-Scholes Gamma and GEX


In [4]:
def _scalar_or_array(values):
    arr = np.asarray(values)
    if np.ndim(arr) == 0:
        return float(arr)
    return arr


def _bs_valid_mask(S_arr, K_arr, T_arr, sigma_arr):
    return (S_arr > 0) & (K_arr > 0) & (T_arr > 0) & (sigma_arr > 0)


def _bs_d1(S_arr, K_arr, T_arr, sigma_arr, r=0.045, q=0.0):
    return (np.log(S_arr / K_arr) + (r - q + 0.5 * sigma_arr**2) * T_arr) / (sigma_arr * np.sqrt(T_arr))


def bs_delta(S, K, T, sigma, option_type="call", r=0.045, q=0.0):
    S_arr = np.asarray(S, dtype=float)
    K_arr = np.asarray(K, dtype=float)
    T_arr = np.asarray(T, dtype=float)
    sigma_arr = np.asarray(sigma, dtype=float)
    shape = np.broadcast(S_arr, K_arr, T_arr, sigma_arr).shape
    delta = np.full(shape, np.nan, dtype=float)
    option_arr = np.broadcast_to(np.asarray(option_type), shape).astype(str)

    valid = _bs_valid_mask(S_arr, K_arr, T_arr, sigma_arr)
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        d1 = _bs_d1(S_arr, K_arr, T_arr, sigma_arr, r=r, q=q)
        call_delta = np.exp(-q * T_arr) * norm.cdf(d1)
        put_delta = np.exp(-q * T_arr) * (norm.cdf(d1) - 1.0)
        calc = np.where(np.char.lower(option_arr) == "call", call_delta, put_delta)
        delta = np.where(valid, calc, np.nan)

    return _scalar_or_array(delta)


def bs_gamma(S, K, T, sigma, r=0.045, q=0.0):
    S_arr = np.asarray(S, dtype=float)
    K_arr = np.asarray(K, dtype=float)
    T_arr = np.asarray(T, dtype=float)
    sigma_arr = np.asarray(sigma, dtype=float)

    valid = _bs_valid_mask(S_arr, K_arr, T_arr, sigma_arr)
    gamma = np.full(np.broadcast(S_arr, K_arr, T_arr, sigma_arr).shape, np.nan, dtype=float)

    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        d1 = _bs_d1(S_arr, K_arr, T_arr, sigma_arr, r=r, q=q)
        calc = np.exp(-q * T_arr) * norm.pdf(d1) / (S_arr * sigma_arr * np.sqrt(T_arr))
        gamma = np.where(valid, calc, np.nan)

    return _scalar_or_array(gamma)


def bs_vanna_1vol(S, K, T, sigma, option_type="call", r=0.045, q=0.0, vol_shift=0.01):
    sigma_arr = np.asarray(sigma, dtype=float)
    delta_now = bs_delta(S, K, T, sigma_arr, option_type=option_type, r=r, q=q)
    delta_up = bs_delta(S, K, T, sigma_arr + vol_shift, option_type=option_type, r=r, q=q)
    out = np.asarray(delta_up, dtype=float) - np.asarray(delta_now, dtype=float)
    return _scalar_or_array(out)


def bs_charm_1d(S, K, T, sigma, option_type="call", r=0.045, q=0.0, day_count=365.0):
    T_arr = np.asarray(T, dtype=float)
    delta_now = bs_delta(S, K, T_arr, sigma, option_type=option_type, r=r, q=q)
    delta_next = bs_delta(S, K, T_arr - 1.0 / day_count, sigma, option_type=option_type, r=r, q=q)
    out = np.asarray(delta_next, dtype=float) - np.asarray(delta_now, dtype=float)
    out = np.where(T_arr > 1.0 / day_count, out, np.nan)
    return _scalar_or_array(out)


def add_gamma_and_gex(chain_df) -> pd.DataFrame:
    greek_cols = [
        "delta",
        "gamma",
        "vanna_1vol",
        "charm_1d",
        "raw_gex",
        "signed_gex",
        "gex",
        "raw_vanna_exposure",
        "signed_vanna_exposure",
        "raw_charm_exposure",
        "signed_charm_exposure",
    ]
    if chain_df is None or chain_df.empty:
        out = pd.DataFrame() if chain_df is None else chain_df.copy()
        for col in greek_cols:
            out[col] = np.nan
        return out

    out = chain_df.copy()
    S = out["spot"].to_numpy()
    K = out["strike"].to_numpy()
    T = out["T"].to_numpy()
    sigma = out["impliedVolatility"].to_numpy()
    option_type = out["option_type"].to_numpy()

    out["delta"] = bs_delta(S, K, T, sigma, option_type=option_type, r=RISK_FREE_RATE, q=DIVIDEND_YIELD)
    out["gamma"] = bs_gamma(S, K, T, sigma, r=RISK_FREE_RATE, q=DIVIDEND_YIELD)
    out["vanna_1vol"] = bs_vanna_1vol(S, K, T, sigma, option_type=option_type, r=RISK_FREE_RATE, q=DIVIDEND_YIELD)
    out["charm_1d"] = bs_charm_1d(S, K, T, sigma, option_type=option_type, r=RISK_FREE_RATE, q=DIVIDEND_YIELD)

    open_interest = out["openInterest"].fillna(0)
    option_sign = np.where(out["option_type"].eq("call"), 1.0, -1.0)
    contract_delta_notional = open_interest * 100.0 * out["spot"]

    out["raw_gex"] = out["gamma"] * open_interest * 100.0 * out["spot"] ** 2 * 0.01
    out["signed_gex"] = option_sign * out["raw_gex"]
    out["gex"] = out["signed_gex"]
    out["raw_vanna_exposure"] = out["vanna_1vol"] * contract_delta_notional
    out["signed_vanna_exposure"] = option_sign * out["raw_vanna_exposure"]
    out["raw_charm_exposure"] = out["charm_1d"] * contract_delta_notional
    out["signed_charm_exposure"] = option_sign * out["raw_charm_exposure"]
    return out


## 6. Gamma Flip / Wall / Short-Dated GEX


In [5]:
def recompute_signed_gex_at_spot(chain_df, scenario_spot) -> float:
    if chain_df is None or chain_df.empty or pd.isna(scenario_spot) or scenario_spot <= 0:
        return np.nan
    gamma = bs_gamma(
        scenario_spot,
        chain_df["strike"].to_numpy(),
        chain_df["T"].to_numpy(),
        chain_df["impliedVolatility"].to_numpy(),
        r=RISK_FREE_RATE,
        q=DIVIDEND_YIELD,
    )
    raw = gamma * chain_df["openInterest"].fillna(0).to_numpy() * 100.0 * scenario_spot**2 * 0.01
    signed = np.where(chain_df["option_type"].eq("call").to_numpy(), raw, -raw)
    return float(np.nansum(signed))


def estimate_gamma_flip(chain_df, lower_mult=0.80, upper_mult=1.20, n_grid=161) -> tuple[float, float, pd.DataFrame]:
    if chain_df is None or chain_df.empty:
        return np.nan, np.nan, pd.DataFrame(columns=["scenario_spot", "scenario_net_gex"])
    spot = _safe_last(chain_df["spot"])
    if pd.isna(spot) or spot <= 0:
        return np.nan, np.nan, pd.DataFrame(columns=["scenario_spot", "scenario_net_gex"])

    grid = np.linspace(spot * lower_mult, spot * upper_mult, n_grid)
    net = np.array([recompute_signed_gex_at_spot(chain_df, s) for s in grid], dtype=float)
    gex_grid = pd.DataFrame({"scenario_spot": grid, "scenario_net_gex": net})

    candidates = []
    for i in range(len(grid) - 1):
        y0, y1 = net[i], net[i + 1]
        if not np.isfinite(y0) or not np.isfinite(y1):
            continue
        if y0 == 0:
            candidates.append(grid[i])
        elif y0 * y1 < 0:
            x0, x1 = grid[i], grid[i + 1]
            candidates.append(x0 - y0 * (x1 - x0) / (y1 - y0))

    if candidates:
        gamma_flip = min(candidates, key=lambda x: abs(x - spot))
    elif np.isfinite(net).any():
        gamma_flip = grid[np.nanargmin(np.abs(net))]
    else:
        gamma_flip = np.nan

    spot_to_flip_pct = (gamma_flip - spot) / spot if pd.notna(gamma_flip) and spot != 0 else np.nan
    return float(gamma_flip), float(spot_to_flip_pct), gex_grid


def _empty_option_summary(symbol=None) -> dict:
    return {
        "symbol": symbol,
        "spot": np.nan,
        "net_gex": 0.0,
        "abs_gex": 0.0,
        "gamma_flip": np.nan,
        "spot_to_flip_pct": np.nan,
        "call_wall": np.nan,
        "put_wall": np.nan,
        "spot_to_call_wall_pct": np.nan,
        "spot_to_put_wall_pct": np.nan,
        "near_2pct_gex": 0.0,
        "near_2pct_abs_gex": 0.0,
        "near_5pct_gex": 0.0,
        "near_5pct_abs_gex": 0.0,
        "short_7d_gex": 0.0,
        "short_7d_abs_gex": 0.0,
        "short_14d_gex": 0.0,
        "short_14d_abs_gex": 0.0,
        "near_5pct_abs_gex_ratio": 0.0,
        "short_7d_abs_gex_ratio": 0.0,
        "short_14d_abs_gex_ratio": 0.0,
        "call_oi": 0.0,
        "put_oi": 0.0,
        "call_volume": 0.0,
        "put_volume": 0.0,
        "put_call_oi_ratio": np.nan,
        "put_call_volume_ratio": np.nan,
        "atm_iv": np.nan,
    }


def summarize_option_gamma_extended(chain_df) -> tuple[dict, pd.DataFrame]:
    if chain_df is None or chain_df.empty:
        return _empty_option_summary(), pd.DataFrame(columns=["scenario_spot", "scenario_net_gex"])

    chain = add_gamma_and_gex(chain_df)
    symbol = _safe_last(chain["symbol"])
    spot = _safe_last(chain["spot"])
    if pd.isna(spot) or spot <= 0:
        return _empty_option_summary(symbol), pd.DataFrame(columns=["scenario_spot", "scenario_net_gex"])

    net_gex = float(chain["signed_gex"].sum(skipna=True))
    abs_gex = float(chain["signed_gex"].abs().sum(skipna=True))
    gamma_flip, spot_to_flip_pct, gex_grid = estimate_gamma_flip(chain)

    calls = chain[chain["option_type"].eq("call")]
    puts = chain[chain["option_type"].eq("put")]
    call_wall = np.nan
    put_wall = np.nan
    if not calls.empty and calls["raw_gex"].notna().any():
        call_wall = float(calls.groupby("strike")["raw_gex"].sum().idxmax())
    if not puts.empty and puts["raw_gex"].notna().any():
        put_wall = float(puts.groupby("strike")["raw_gex"].sum().idxmax())

    near_2 = chain[(chain["strike"] >= spot * 0.98) & (chain["strike"] <= spot * 1.02)]
    near_5 = chain[(chain["strike"] >= spot * 0.95) & (chain["strike"] <= spot * 1.05)]
    short_7 = chain[(chain["dte"] >= 0) & (chain["dte"] <= 7)]
    short_14 = chain[(chain["dte"] >= 0) & (chain["dte"] <= 14)]

    call_oi = float(calls["openInterest"].sum(skipna=True))
    put_oi = float(puts["openInterest"].sum(skipna=True))
    call_volume = float(calls["volume"].sum(skipna=True))
    put_volume = float(puts["volume"].sum(skipna=True))

    atm_idx = (chain["strike"] - spot).abs().sort_values().index[: max(1, min(6, len(chain)))]
    atm_iv = float(chain.loc[atm_idx, "impliedVolatility"].median()) if len(atm_idx) else np.nan

    summary = {
        "symbol": symbol,
        "spot": float(spot),
        "net_gex": net_gex,
        "abs_gex": abs_gex,
        "gamma_flip": gamma_flip,
        "spot_to_flip_pct": spot_to_flip_pct,
        "call_wall": call_wall,
        "put_wall": put_wall,
        "spot_to_call_wall_pct": (call_wall - spot) / spot if pd.notna(call_wall) else np.nan,
        "spot_to_put_wall_pct": (put_wall - spot) / spot if pd.notna(put_wall) else np.nan,
        "near_2pct_gex": float(near_2["signed_gex"].sum(skipna=True)),
        "near_2pct_abs_gex": float(near_2["signed_gex"].abs().sum(skipna=True)),
        "near_5pct_gex": float(near_5["signed_gex"].sum(skipna=True)),
        "near_5pct_abs_gex": float(near_5["signed_gex"].abs().sum(skipna=True)),
        "short_7d_gex": float(short_7["signed_gex"].sum(skipna=True)),
        "short_7d_abs_gex": float(short_7["signed_gex"].abs().sum(skipna=True)),
        "short_14d_gex": float(short_14["signed_gex"].sum(skipna=True)),
        "short_14d_abs_gex": float(short_14["signed_gex"].abs().sum(skipna=True)),
        "near_5pct_abs_gex_ratio": float(near_5["signed_gex"].abs().sum(skipna=True) / abs_gex) if abs_gex else 0.0,
        "short_7d_abs_gex_ratio": float(short_7["signed_gex"].abs().sum(skipna=True) / abs_gex) if abs_gex else 0.0,
        "short_14d_abs_gex_ratio": float(short_14["signed_gex"].abs().sum(skipna=True) / abs_gex) if abs_gex else 0.0,
        "call_oi": call_oi,
        "put_oi": put_oi,
        "call_volume": call_volume,
        "put_volume": put_volume,
        "put_call_oi_ratio": put_oi / call_oi if call_oi else np.nan,
        "put_call_volume_ratio": put_volume / call_volume if call_volume else np.nan,
        "atm_iv": atm_iv,
    }
    return summary, gex_grid


def _empty_second_order_summary(symbol=None) -> dict:
    return {
        "symbol": symbol,
        "spot": np.nan,
        "net_vanna_exposure": 0.0,
        "abs_vanna_exposure": 0.0,
        "net_charm_exposure": 0.0,
        "abs_charm_exposure": 0.0,
        "near_5pct_vanna_exposure": 0.0,
        "near_5pct_charm_exposure": 0.0,
        "short_7d_vanna_exposure": 0.0,
        "short_7d_charm_exposure": 0.0,
        "near_5pct_abs_vanna_exposure": 0.0,
        "near_5pct_abs_charm_exposure": 0.0,
        "short_7d_abs_vanna_exposure": 0.0,
        "short_7d_abs_charm_exposure": 0.0,
        "vanna_near_5pct_abs_ratio": 0.0,
        "vanna_short_7d_abs_ratio": 0.0,
        "charm_near_5pct_abs_ratio": 0.0,
        "charm_short_7d_abs_ratio": 0.0,
        "vanna_abs_concentration_ratio": 0.0,
        "charm_abs_concentration_ratio": 0.0,
    }


def summarize_second_order_greeks(chain_df) -> dict:
    if chain_df is None or chain_df.empty:
        return _empty_second_order_summary()

    chain = add_gamma_and_gex(chain_df)
    symbol = _safe_last(chain["symbol"])
    spot = _safe_last(chain["spot"])
    if pd.isna(spot) or spot <= 0:
        return _empty_second_order_summary(symbol)

    near_5 = chain[(chain["strike"] >= spot * 0.95) & (chain["strike"] <= spot * 1.05)]
    short_7 = chain[(chain["dte"] >= 0) & (chain["dte"] <= 7)]

    abs_vanna = float(chain["signed_vanna_exposure"].abs().sum(skipna=True))
    abs_charm = float(chain["signed_charm_exposure"].abs().sum(skipna=True))
    near_abs_vanna = float(near_5["signed_vanna_exposure"].abs().sum(skipna=True))
    near_abs_charm = float(near_5["signed_charm_exposure"].abs().sum(skipna=True))
    short_abs_vanna = float(short_7["signed_vanna_exposure"].abs().sum(skipna=True))
    short_abs_charm = float(short_7["signed_charm_exposure"].abs().sum(skipna=True))

    vanna_near_ratio = near_abs_vanna / abs_vanna if abs_vanna else 0.0
    vanna_short_ratio = short_abs_vanna / abs_vanna if abs_vanna else 0.0
    charm_near_ratio = near_abs_charm / abs_charm if abs_charm else 0.0
    charm_short_ratio = short_abs_charm / abs_charm if abs_charm else 0.0

    return {
        "symbol": symbol,
        "spot": float(spot),
        "net_vanna_exposure": float(chain["signed_vanna_exposure"].sum(skipna=True)),
        "abs_vanna_exposure": abs_vanna,
        "net_charm_exposure": float(chain["signed_charm_exposure"].sum(skipna=True)),
        "abs_charm_exposure": abs_charm,
        "near_5pct_vanna_exposure": float(near_5["signed_vanna_exposure"].sum(skipna=True)),
        "near_5pct_charm_exposure": float(near_5["signed_charm_exposure"].sum(skipna=True)),
        "short_7d_vanna_exposure": float(short_7["signed_vanna_exposure"].sum(skipna=True)),
        "short_7d_charm_exposure": float(short_7["signed_charm_exposure"].sum(skipna=True)),
        "near_5pct_abs_vanna_exposure": near_abs_vanna,
        "near_5pct_abs_charm_exposure": near_abs_charm,
        "short_7d_abs_vanna_exposure": short_abs_vanna,
        "short_7d_abs_charm_exposure": short_abs_charm,
        "vanna_near_5pct_abs_ratio": float(vanna_near_ratio),
        "vanna_short_7d_abs_ratio": float(vanna_short_ratio),
        "charm_near_5pct_abs_ratio": float(charm_near_ratio),
        "charm_short_7d_abs_ratio": float(charm_short_ratio),
        "vanna_abs_concentration_ratio": float(max(vanna_near_ratio, vanna_short_ratio)),
        "charm_abs_concentration_ratio": float(max(charm_near_ratio, charm_short_ratio)),
    }


def _vanna_charm_risk_comment(row: pd.Series) -> str:
    high_vanna = bool(row.get("high_vanna_risk", False))
    high_charm = bool(row.get("high_charm_risk", False))
    net_vanna = row.get("net_vanna_exposure", np.nan)
    net_charm = row.get("net_charm_exposure", np.nan)
    vanna_side = "正" if pd.notna(net_vanna) and net_vanna >= 0 else "負"
    charm_side = "正" if pd.notna(net_charm) and net_charm >= 0 else "負"
    if high_vanna and high_charm:
        return f"Vanna/Charmの集中が高い。IV変化と1日時間経過でdealer delta調整が出やすい（net Vanna {vanna_side}, net Charm {charm_side}）。"
    if high_vanna:
        return f"Vanna集中が高い。IV変化に伴うdealer delta調整に注意（net Vanna {vanna_side}）。"
    if high_charm:
        return f"Charm集中が高い。時間経過に伴うdealer delta調整に注意（net Charm {charm_side}）。"
    return "Vanna/Charmの集中は相対的に限定的。Gamma/GEX水準を主軸に判断。"


def add_vanna_charm_flags(second_order_df: pd.DataFrame) -> pd.DataFrame:
    if second_order_df is None or second_order_df.empty:
        return pd.DataFrame()

    out = second_order_df.copy()
    vanna_abs = pd.to_numeric(out["abs_vanna_exposure"], errors="coerce").fillna(0)
    charm_abs = pd.to_numeric(out["abs_charm_exposure"], errors="coerce").fillna(0)
    vanna_threshold = vanna_abs.quantile(0.75) if len(vanna_abs) else np.inf
    charm_threshold = charm_abs.quantile(0.75) if len(charm_abs) else np.inf

    out["high_vanna_risk"] = ((vanna_abs >= vanna_threshold) & (out["vanna_abs_concentration_ratio"].fillna(0) >= 0.25)) | (out["vanna_abs_concentration_ratio"].fillna(0) >= 0.50)
    out["high_charm_risk"] = ((charm_abs >= charm_threshold) & (out["charm_abs_concentration_ratio"].fillna(0) >= 0.25)) | (out["charm_abs_concentration_ratio"].fillna(0) >= 0.50)
    out["vanna_charm_risk_comment"] = out.apply(_vanna_charm_risk_comment, axis=1)
    return out


## 7. Scoring


In [6]:
def _zscore(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    mean = s.mean(skipna=True)
    std = s.std(skipna=True, ddof=0)
    if pd.isna(std) or std == 0:
        return pd.Series(0.0, index=series.index)
    return (s - mean) / std


def _between(value, low, high) -> bool:
    return bool(pd.notna(value) and low < value < high)


def compute_option_scores_extended(summary_df: pd.DataFrame) -> pd.DataFrame:
    if summary_df is None or summary_df.empty:
        return pd.DataFrame()
    out = summary_df.copy()

    gamma_scores = []
    for _, row in out.iterrows():
        score = 0.0
        if pd.notna(row.get("net_gex")):
            score += 0.75 if row["net_gex"] > 0 else -0.75 if row["net_gex"] < 0 else 0.0
        if pd.notna(row.get("near_5pct_gex")):
            score += 0.50 if row["near_5pct_gex"] > 0 else -0.50 if row["near_5pct_gex"] < 0 else 0.0
        if pd.notna(row.get("spot_to_flip_pct")) and abs(row["spot_to_flip_pct"]) < 0.02:
            score -= 0.75
        if pd.notna(row.get("spot")) and pd.notna(row.get("gamma_flip")):
            if row["spot"] < row["gamma_flip"]:
                score -= 0.75
            elif row["spot"] > row["gamma_flip"]:
                score += 0.25
        if _between(row.get("spot_to_call_wall_pct"), 0, 0.02):
            score -= 0.50
        if pd.notna(row.get("spot_to_call_wall_pct")) and row["spot_to_call_wall_pct"] < 0:
            score += 0.25
        if _between(row.get("spot_to_put_wall_pct"), -0.02, 0):
            score -= 0.50
        if pd.notna(row.get("spot")) and pd.notna(row.get("put_wall")) and row["spot"] < row["put_wall"]:
            score -= 1.00
        if pd.notna(row.get("near_5pct_abs_gex_ratio")) and row["near_5pct_abs_gex_ratio"] > 0.50:
            score -= 0.25
        if pd.notna(row.get("short_7d_abs_gex_ratio")) and row["short_7d_abs_gex_ratio"] > 0.35:
            score -= 0.50
        if pd.notna(row.get("short_14d_abs_gex_ratio")) and row["short_14d_abs_gex_ratio"] > 0.55:
            score -= 0.25
        if pd.notna(row.get("short_7d_gex")):
            score += -0.25 if row["short_7d_gex"] < 0 else 0.25 if row["short_7d_gex"] > 0 else 0.0
        gamma_scores.append(score)

    out["gamma_score"] = gamma_scores
    out["z_pcr_oi"] = _zscore(out["put_call_oi_ratio"])
    out["z_pcr_vol"] = _zscore(out["put_call_volume_ratio"])
    out["z_atm_iv"] = _zscore(out["atm_iv"])

    option_scores = []
    for _, row in out.iterrows():
        score = 0.0
        z_pcr_oi = row.get("z_pcr_oi")
        z_pcr_vol = row.get("z_pcr_vol")
        z_atm_iv = row.get("z_atm_iv")
        if pd.notna(z_pcr_oi) and z_pcr_oi < -1.0:
            score -= 0.50
        if pd.notna(z_pcr_oi) and z_pcr_oi > 1.0:
            score -= 0.50
        if pd.notna(z_pcr_oi) and -0.5 <= z_pcr_oi <= 0.5:
            score += 0.25
        if pd.notna(z_pcr_vol) and z_pcr_vol > 1.0:
            score -= 0.50
        if pd.notna(z_atm_iv) and z_atm_iv > 1.0:
            score -= 0.25
        option_scores.append(score)
    out["option_demand_score"] = option_scores
    return out


def compute_total_scores(price_signals: pd.DataFrame, option_scores: pd.DataFrame) -> pd.DataFrame:
    base = price_signals.copy()
    if option_scores is not None and not option_scores.empty:
        base = base.merge(option_scores, on="symbol", how="left")

    for col in ["gamma_score", "option_demand_score"]:
        if col not in base:
            base[col] = 0.0
        base[col] = base[col].fillna(0.0)

    base["risk_control_score"] = 0.0
    base["risk_control_score"] += np.where(base["above_20ma"].fillna(False), 0.50, -1.00)
    base["risk_control_score"] += np.where(~base["above_50ma"].fillna(False), -0.50, 0.00)
    base["risk_control_score"] += np.where(base["drawdown_gt_7pct"].fillna(False), -1.00, 0.00)

    base["total_score"] = (
        0.40 * base["momentum_score"].fillna(0)
        + 0.30 * base["gamma_score"].fillna(0)
        + 0.20 * base["option_demand_score"].fillna(0)
        + 0.10 * base["risk_control_score"].fillna(0)
    )

    base["positive_gamma_regime"] = base.get("net_gex", pd.Series(index=base.index, dtype=float)).fillna(0) > 0
    base["below_gamma_flip"] = base.apply(
        lambda r: bool(pd.notna(r.get("spot")) and pd.notna(r.get("gamma_flip")) and r["spot"] < r["gamma_flip"]),
        axis=1,
    )
    base["near_gamma_flip"] = base.get("spot_to_flip_pct", pd.Series(index=base.index, dtype=float)).abs() < 0.02
    base["near_call_wall"] = base.get("spot_to_call_wall_pct", pd.Series(index=base.index, dtype=float)).apply(lambda x: _between(x, 0, 0.02))
    base["near_put_wall"] = base.get("spot_to_put_wall_pct", pd.Series(index=base.index, dtype=float)).apply(lambda x: _between(x, -0.02, 0))
    base["put_wall_broken"] = base.apply(
        lambda r: bool(pd.notna(r.get("spot")) and pd.notna(r.get("put_wall")) and r["spot"] < r["put_wall"]),
        axis=1,
    )
    base["short_dated_risk"] = base.get("short_7d_abs_gex_ratio", pd.Series(index=base.index, dtype=float)).fillna(0) > 0.35
    base["overweight_allowed"] = ~(
        (~base["above_20ma"].fillna(False))
        | base["drawdown_gt_7pct"].fillna(False)
        | base["put_wall_broken"].fillna(False)
        | (base["below_gamma_flip"].fillna(False) & (base["momentum_score"].fillna(0) < 1.0))
        | base["near_gamma_flip"].fillna(False)
        | base["near_call_wall"].fillna(False)
        | base["short_dated_risk"].fillna(False)
    )

    base["sector_name"] = base["symbol"].map(SECTOR_NAMES).fillna(base["symbol"])
    return base.sort_values("total_score", ascending=False).reset_index(drop=True)


## 7.1 IV Expected Move Overlay


In [7]:
def _level_inside_expected_move(row: pd.Series, level_col: str) -> bool:
    level = row.get(level_col)
    low = row.get("expected_low")
    high = row.get("expected_high")
    return bool(pd.notna(level) and pd.notna(low) and pd.notna(high) and low <= level <= high)


def _expected_move_to_level_ratio(row: pd.Series, level_col: str) -> float:
    level = row.get(level_col)
    spot = row.get("spot")
    expected_move_abs = row.get("expected_move_abs")
    if pd.isna(level) or pd.isna(spot) or pd.isna(expected_move_abs) or expected_move_abs <= 0:
        return np.nan
    distance = abs(level - spot)
    if distance == 0:
        return np.inf
    return float(expected_move_abs / distance)


def compute_iv_expected_moves(portfolio_df: pd.DataFrame) -> pd.DataFrame:
    if portfolio_df is None or portfolio_df.empty:
        return pd.DataFrame()

    out = portfolio_df.copy()
    out["expected_move_days"] = IV_EXPECTED_MOVE_DAYS
    out["expected_move_z"] = IV_EXPECTED_MOVE_Z
    out["expected_move_pct"] = out["atm_iv"].astype(float) * math.sqrt(IV_EXPECTED_MOVE_DAYS / 365.0) * IV_EXPECTED_MOVE_Z
    out["expected_move_abs"] = out["spot"].astype(float) * out["expected_move_pct"]
    out["expected_low"] = out["spot"] - out["expected_move_abs"]
    out["expected_high"] = out["spot"] + out["expected_move_abs"]

    out["call_wall_inside_expected_move"] = out.apply(lambda row: _level_inside_expected_move(row, "call_wall"), axis=1)
    out["put_wall_inside_expected_move"] = out.apply(lambda row: _level_inside_expected_move(row, "put_wall"), axis=1)
    out["gamma_flip_inside_expected_move"] = out.apply(lambda row: _level_inside_expected_move(row, "gamma_flip"), axis=1)

    out["expected_move_to_call_wall_ratio"] = out.apply(lambda row: _expected_move_to_level_ratio(row, "call_wall"), axis=1)
    out["expected_move_to_put_wall_ratio"] = out.apply(lambda row: _expected_move_to_level_ratio(row, "put_wall"), axis=1)
    out["expected_move_to_gamma_flip_ratio"] = out.apply(lambda row: _expected_move_to_level_ratio(row, "gamma_flip"), axis=1)
    return out


def _iv_weekly_comment(row: pd.Series) -> str:
    if pd.isna(row.get("expected_move_pct")):
        return "ATM IVが不足しており、週次予想変動は評価できない。"

    comments = [
        f"7日1σ予想変動は±{row['expected_move_pct'] * 100:.1f}%（±{row['expected_move_abs']:.2f}）。"
    ]
    if row.get("gamma_flip_inside_expected_move", False):
        comments.append("Gamma Flipが予想レンジ内にあり、週内にレジーム転換を試す可能性。")
    if row.get("call_wall_inside_expected_move", False):
        comments.append("Call Wallが予想レンジ内にあり、上値抵抗に到達しうる。")
    if row.get("put_wall_inside_expected_move", False):
        comments.append("Put Wallが予想レンジ内にあり、下値支持/割れリスクを確認したい。")
    if not any(
        [
            row.get("gamma_flip_inside_expected_move", False),
            row.get("call_wall_inside_expected_move", False),
            row.get("put_wall_inside_expected_move", False),
        ]
    ):
        if row.get("net_gex", 0) > 0:
            comments.append("主要水準は予想レンジ外で、Net GEXも正のため週次リスクは相対的に抑制的。")
        else:
            comments.append("主要水準は予想レンジ外だが、Net GEXは負で値動き拡大には注意。")
    if pd.notna(row.get("z_atm_iv")) and row["z_atm_iv"] > 1.0:
        comments.append("ATM IVはクロスセクションで高く、イベント/不安定性を織り込んでいる。")
    return "".join(comments)


def compute_iv_overlay_scores(portfolio_df: pd.DataFrame) -> pd.DataFrame:
    out = compute_iv_expected_moves(portfolio_df)
    if out.empty:
        return out

    iv_scores = []
    for _, row in out.iterrows():
        score = 0.0
        if row.get("gamma_flip_inside_expected_move", False):
            score -= 0.75
        if row.get("call_wall_inside_expected_move", False):
            score -= 0.50
        if row.get("put_wall_inside_expected_move", False):
            score -= 0.50
        if pd.notna(row.get("z_atm_iv")) and row["z_atm_iv"] > 1.0:
            score -= 0.25
        if (
            not row.get("gamma_flip_inside_expected_move", False)
            and not row.get("call_wall_inside_expected_move", False)
            and not row.get("put_wall_inside_expected_move", False)
            and row.get("net_gex", 0) > 0
        ):
            score += 0.25
        iv_scores.append(score)

    out["iv_risk_score"] = iv_scores
    out["iv_adjusted_score"] = out["total_score"].fillna(0) + 0.20 * out["iv_risk_score"].fillna(0)
    out["iv_adjusted_rank"] = out["iv_adjusted_score"].rank(ascending=False, method="first").astype(int)
    out["iv_weekly_comment"] = out.apply(_iv_weekly_comment, axis=1)
    return out


def make_iv_expected_move_table(portfolio_df: pd.DataFrame) -> pd.DataFrame:
    cols = [
        "symbol",
        "sector_name",
        "spot",
        "atm_iv",
        "expected_move_days",
        "expected_move_pct",
        "expected_move_abs",
        "expected_low",
        "expected_high",
        "gamma_flip",
        "call_wall",
        "put_wall",
        "gamma_flip_inside_expected_move",
        "call_wall_inside_expected_move",
        "put_wall_inside_expected_move",
        "expected_move_to_gamma_flip_ratio",
        "expected_move_to_call_wall_ratio",
        "expected_move_to_put_wall_ratio",
        "iv_risk_score",
        "total_score",
        "iv_adjusted_score",
        "proposed_weight",
        "active_weight",
        "iv_weekly_comment",
    ]
    available = [c for c in cols if c in portfolio_df.columns]
    return portfolio_df[available].copy()


## 8. Portfolio Construction


In [8]:
def construct_portfolio(scores_df: pd.DataFrame, benchmark_weights: dict[str, float] = BENCHMARK_WEIGHTS) -> pd.DataFrame:
    if scores_df is None or scores_df.empty:
        return pd.DataFrame()

    out = scores_df.copy()
    out["benchmark_weight"] = out["symbol"].map(benchmark_weights).astype(float)
    out = out.dropna(subset=["benchmark_weight"]).copy()
    bm_sum = out["benchmark_weight"].sum()
    if bm_sum > 0:
        out["benchmark_weight"] = out["benchmark_weight"] / bm_sum

    active = pd.Series(0.0, index=out.index)
    top_allowed = out[out["overweight_allowed"].fillna(False)].sort_values("total_score", ascending=False).head(3)
    bottom = out.sort_values("total_score", ascending=True).head(3)

    positive_values = [0.020, 0.015, 0.010][: len(top_allowed)]
    negative_values = [-0.020, -0.015, -0.010][: len(bottom)]

    for idx, value in zip(top_allowed.index, positive_values):
        active.loc[idx] += value
    for idx, value in zip(bottom.index, negative_values):
        active.loc[idx] += value

    pos_total = active[active > 0].sum()
    neg_total = -active[active < 0].sum()
    if pos_total > 0 and neg_total > 0:
        if pos_total > neg_total:
            active.loc[active > 0] *= neg_total / pos_total
        elif neg_total > pos_total:
            active.loc[active < 0] *= pos_total / neg_total
    elif pos_total == 0 or neg_total == 0:
        active[:] = 0.0

    raw_proposed = (out["benchmark_weight"] + active).clip(lower=0)
    if raw_proposed.sum() > 0:
        proposed = raw_proposed / raw_proposed.sum()
    else:
        proposed = out["benchmark_weight"] / out["benchmark_weight"].sum()

    out["proposed_weight"] = proposed
    out["active_weight"] = out["proposed_weight"] - out["benchmark_weight"]
    out["rank_total_score"] = out["total_score"].rank(ascending=False, method="first").astype(int)
    out["gex_regime_comment"] = out.apply(explain_sector_gex, axis=1)
    return out.sort_values("total_score", ascending=False).reset_index(drop=True)


def _fmt_pct(value, digits=1) -> str:
    if pd.isna(value):
        return "NA"
    return f"{value * 100:.{digits}f}%"


def explain_sector_gex(row: pd.Series) -> str:
    comments = []
    if pd.isna(row.get("net_gex")):
        comments.append("オプションデータが不足しており、GEXレジームは中立扱い。")
    elif row["net_gex"] > 0:
        comments.append("Net GEXは正で、価格変動は抑制されやすい。")
    elif row["net_gex"] < 0:
        comments.append("Net GEXは負で、ヘッジフロー由来の不安定化に注意。")
    else:
        comments.append("Net GEXは概ね中立。")

    if pd.notna(row.get("spot")) and pd.notna(row.get("gamma_flip")):
        if row["spot"] >= row["gamma_flip"]:
            comments.append("現値はGamma Flipを上回る。")
        else:
            comments.append("現値はGamma Flipを下回り、下方向加速リスクを警戒。")
    if pd.notna(row.get("spot_to_call_wall_pct")):
        if 0 < row["spot_to_call_wall_pct"] < 0.02:
            comments.append("Call Wall直下にあり、新規買いは抑制。")
        elif row["spot_to_call_wall_pct"] >= 0:
            comments.append(f"Call Wallまでは{_fmt_pct(row['spot_to_call_wall_pct'])}。")
        else:
            comments.append("Call Wallを上抜けている。")
    if pd.notna(row.get("spot_to_put_wall_pct")):
        if row.get("put_wall_broken", False):
            comments.append("Put Wallを下回っており、下方向加速リスクが高い。")
        elif -0.02 < row["spot_to_put_wall_pct"] < 0:
            comments.append("Put Wall近傍で、割れに注意。")
    if pd.notna(row.get("short_7d_abs_gex_ratio")):
        if row["short_7d_abs_gex_ratio"] > 0.35:
            comments.append("Short-Dated GEX集中が大きく、今週のイベント/OPEXリスクが高い。")
        else:
            comments.append("Short-Dated GEX集中は限定的。")
    return "".join(comments)


## 9. Dashboard


In [9]:
def plot_allocation(portfolio_df: pd.DataFrame):
    df = portfolio_df.sort_values("proposed_weight", ascending=True)
    fig = go.Figure()
    fig.add_bar(y=df["symbol"], x=df["benchmark_weight"], name="Benchmark", orientation="h")
    fig.add_bar(y=df["symbol"], x=df["proposed_weight"], name="Proposed", orientation="h")
    fig.update_layout(
        title="Benchmark vs Proposed Weight",
        xaxis_title="Weight",
        yaxis_title="Sector ETF",
        barmode="group",
        xaxis_tickformat=".1%",
        height=520,
    )
    return fig


def plot_active_weights(portfolio_df: pd.DataFrame):
    df = portfolio_df.sort_values("active_weight", ascending=True)
    colors = np.where(df["active_weight"] >= 0, "#1f77b4", "#d62728")
    fig = go.Figure()
    fig.add_bar(y=df["symbol"], x=df["active_weight"], orientation="h", marker_color=colors)
    fig.add_vline(x=0, line_width=1, line_color="black")
    fig.update_layout(title="Active Tilt", xaxis_title="Active Weight", yaxis_title="Sector ETF", xaxis_tickformat=".1%", height=520)
    return fig


def plot_signal_heatmap(scores_df: pd.DataFrame):
    cols = ["momentum_score", "gamma_score", "option_demand_score", "risk_control_score", "total_score"]
    df = scores_df.sort_values("total_score", ascending=False).set_index("symbol")
    z = df[cols].astype(float)
    fig = go.Figure(data=go.Heatmap(z=z.values, x=cols, y=z.index, colorscale="RdBu", zmid=0, colorbar_title="Score"))
    fig.update_layout(title="Signal Heatmap", height=480)
    return fig


def plot_gex_indicator_heatmap(scores_df: pd.DataFrame):
    cols = [
        "net_gex",
        "abs_gex",
        "spot_to_flip_pct",
        "near_5pct_gex",
        "short_7d_gex",
        "short_14d_gex",
        "spot_to_call_wall_pct",
        "spot_to_put_wall_pct",
    ]
    df = scores_df.sort_values("total_score", ascending=False).set_index("symbol")
    z = df[cols].apply(_zscore).fillna(0)
    fig = go.Figure(data=go.Heatmap(z=z.values, x=cols, y=z.index, colorscale="RdBu", zmid=0, colorbar_title="z-score"))
    fig.update_layout(title="GEX Indicator Heatmap", height=520)
    return fig


def make_gex_decision_table(portfolio_df: pd.DataFrame) -> pd.DataFrame:
    cols = [
        "symbol",
        "sector_name",
        "spot",
        "net_gex",
        "abs_gex",
        "gamma_flip",
        "spot_to_flip_pct",
        "call_wall",
        "put_wall",
        "spot_to_call_wall_pct",
        "spot_to_put_wall_pct",
        "near_5pct_gex",
        "short_7d_gex",
        "short_14d_gex",
        "positive_gamma_regime",
        "below_gamma_flip",
        "near_gamma_flip",
        "near_call_wall",
        "near_put_wall",
        "put_wall_broken",
        "short_dated_risk",
        "gamma_score",
        "option_demand_score",
        "total_score",
        "active_weight",
        "proposed_weight",
        "gex_regime_comment",
        "high_vanna_risk",
        "high_charm_risk",
        "vanna_abs_concentration_ratio",
        "charm_abs_concentration_ratio",
        "vanna_charm_risk_comment",
    ]
    available = [c for c in cols if c in portfolio_df.columns]
    return portfolio_df[available].copy()


def _wall_shapes(row: pd.Series, y0=0, y1=1, yref="paper"):
    shapes = []
    levels = [
        ("spot", "black", "dash"),
        ("gamma_flip", "#9467bd", "dot"),
        ("call_wall", "#2ca02c", "dash"),
        ("put_wall", "#d62728", "dash"),
    ]
    for col, color, dash in levels:
        value = row.get(col)
        if pd.notna(value):
            shapes.append(
                {
                    "type": "line",
                    "xref": "x",
                    "yref": yref,
                    "x0": value,
                    "x1": value,
                    "y0": y0,
                    "y1": y1,
                    "line": {"color": color, "width": 1.5, "dash": dash},
                }
            )
    return shapes


def plot_gex_by_strike(chains_by_symbol: dict[str, pd.DataFrame], summary_df: pd.DataFrame, range_mult=0.20):
    fig = go.Figure()
    buttons = []
    symbols = [s for s in SECTOR_ETFS if s in chains_by_symbol and not chains_by_symbol[s].empty]
    if not symbols:
        fig.update_layout(title="GEX by Strike: no option data")
        return fig

    trace_count = 0
    shapes_by_symbol = {}
    first_symbol = symbols[0]
    for symbol in symbols:
        chain = add_gamma_and_gex(chains_by_symbol[symbol])
        row = summary_df[summary_df["symbol"].eq(symbol)].iloc[0]
        spot = row["spot"]
        view = chain[(chain["strike"] >= spot * (1 - range_mult)) & (chain["strike"] <= spot * (1 + range_mult))]
        agg = (
            view.groupby(["strike", "option_type"], as_index=False)["signed_gex"].sum()
            if not view.empty
            else pd.DataFrame(columns=["strike", "option_type", "signed_gex"])
        )
        calls = agg[agg["option_type"].eq("call")]
        puts = agg[agg["option_type"].eq("put")]
        visible = symbol == first_symbol
        fig.add_bar(x=calls["strike"], y=calls["signed_gex"], name=f"{symbol} Call GEX", marker_color="#1f77b4", visible=visible)
        fig.add_bar(x=puts["strike"], y=puts["signed_gex"], name=f"{symbol} Put GEX", marker_color="#d62728", visible=visible)
        shapes_by_symbol[symbol] = _wall_shapes(row)

        visibility = [False] * (2 * len(symbols))
        visibility[trace_count] = True
        visibility[trace_count + 1] = True
        buttons.append(
            {
                "label": symbol,
                "method": "update",
                "args": [
                    {"visible": visibility},
                    {"title": f"GEX by Strike: {symbol}", "shapes": shapes_by_symbol[symbol]},
                ],
            }
        )
        trace_count += 2

    fig.update_layout(
        title=f"GEX by Strike: {first_symbol}",
        barmode="relative",
        xaxis_title="Strike",
        yaxis_title="Signed GEX",
        shapes=shapes_by_symbol[first_symbol],
        updatemenus=[{"buttons": buttons, "direction": "down", "x": 0.0, "y": 1.15}],
        height=600,
    )
    fig.add_hline(y=0, line_width=1, line_color="black")
    return fig


def plot_gamma_flip_curve(gex_grids: dict[str, pd.DataFrame], summary_df: pd.DataFrame):
    fig = go.Figure()
    buttons = []
    symbols = [s for s in SECTOR_ETFS if s in gex_grids and not gex_grids[s].empty]
    if not symbols:
        fig.update_layout(title="Gamma Flip Curve: no scenario data")
        return fig

    first_symbol = symbols[0]
    shapes_by_symbol = {}
    for i, symbol in enumerate(symbols):
        grid = gex_grids[symbol]
        row = summary_df[summary_df["symbol"].eq(symbol)].iloc[0]
        visible = symbol == first_symbol
        fig.add_scatter(
            x=grid["scenario_spot"],
            y=grid["scenario_net_gex"],
            mode="lines",
            name=symbol,
            visible=visible,
        )
        x0, x1 = grid["scenario_spot"].min(), grid["scenario_spot"].max()
        shapes = [
            {
                "type": "line",
                "xref": "x",
                "yref": "y",
                "x0": x0,
                "x1": x1,
                "y0": 0,
                "y1": 0,
                "line": {"color": "black", "width": 1},
            }
        ] + _wall_shapes(row)
        shapes_by_symbol[symbol] = shapes
        visibility = [False] * len(symbols)
        visibility[i] = True
        buttons.append(
            {
                "label": symbol,
                "method": "update",
                "args": [
                    {"visible": visibility},
                    {"title": f"Gamma Flip Curve: {symbol}", "shapes": shapes_by_symbol[symbol]},
                ],
            }
        )

    fig.update_layout(
        title=f"Gamma Flip Curve: {first_symbol}",
        xaxis_title="Scenario Spot",
        yaxis_title="Scenario Net GEX",
        shapes=shapes_by_symbol[first_symbol],
        updatemenus=[{"buttons": buttons, "direction": "down", "x": 0.0, "y": 1.15}],
        height=560,
    )
    return fig


def plot_price_with_walls(price_df: pd.DataFrame, summary_df: pd.DataFrame):
    fig = go.Figure()
    if price_df is None or price_df.empty:
        fig.update_layout(title="Price + MA + Walls: no price data")
        return fig

    buttons = []
    symbols = [s for s in SECTOR_ETFS if s in set(price_df["symbol"])]
    if not symbols:
        fig.update_layout(title="Price + MA + Walls: no sector price data")
        return fig

    first_symbol = symbols[0]
    shapes_by_symbol = {}
    trace_idx = 0
    for symbol in symbols:
        px_df = price_df[price_df["symbol"].eq(symbol)].sort_values("date").tail(130).copy()
        px_df["ma20"] = px_df["close"].rolling(20, min_periods=20).mean()
        px_df["ma50"] = px_df["close"].rolling(50, min_periods=50).mean()
        visible = symbol == first_symbol
        fig.add_scatter(x=px_df["date"], y=px_df["close"], mode="lines", name=f"{symbol} Close", visible=visible)
        fig.add_scatter(x=px_df["date"], y=px_df["ma20"], mode="lines", name=f"{symbol} 20MA", visible=visible)
        fig.add_scatter(x=px_df["date"], y=px_df["ma50"], mode="lines", name=f"{symbol} 50MA", visible=visible)

        row = summary_df[summary_df["symbol"].eq(symbol)].iloc[0] if symbol in set(summary_df["symbol"]) else pd.Series(dtype=float)
        shapes = []
        if not px_df.empty:
            x0, x1 = px_df["date"].min(), px_df["date"].max()
            for col, color, dash in [
                ("gamma_flip", "#9467bd", "dot"),
                ("call_wall", "#2ca02c", "dash"),
                ("put_wall", "#d62728", "dash"),
            ]:
                value = row.get(col)
                if pd.notna(value):
                    shapes.append(
                        {
                            "type": "line",
                            "xref": "x",
                            "yref": "y",
                            "x0": x0,
                            "x1": x1,
                            "y0": value,
                            "y1": value,
                            "line": {"color": color, "width": 1.5, "dash": dash},
                        }
                    )
        shapes_by_symbol[symbol] = shapes

        visibility = [False] * (3 * len(symbols))
        visibility[trace_idx] = True
        visibility[trace_idx + 1] = True
        visibility[trace_idx + 2] = True
        buttons.append(
            {
                "label": symbol,
                "method": "update",
                "args": [
                    {"visible": visibility},
                    {"title": f"Price + 20MA/50MA + Walls: {symbol}", "shapes": shapes_by_symbol[symbol]},
                ],
            }
        )
        trace_idx += 3

    fig.update_layout(
        title=f"Price + 20MA/50MA + Walls: {first_symbol}",
        xaxis_title="Date",
        yaxis_title="Price",
        shapes=shapes_by_symbol[first_symbol],
        updatemenus=[{"buttons": buttons, "direction": "down", "x": 0.0, "y": 1.15}],
        height=560,
    )
    return fig


## 9.1 IV Overlay Dashboard Functions


In [10]:
def plot_iv_expected_move_bands(iv_df: pd.DataFrame):
    if iv_df is None or iv_df.empty:
        fig = go.Figure()
        fig.update_layout(title="IV Expected Move vs Key Levels: no data")
        return fig

    df = iv_df.sort_values("iv_adjusted_score", ascending=True).copy()
    fig = go.Figure()

    x_lines = []
    y_lines = []
    for _, row in df.iterrows():
        x_lines.extend([row["expected_low"], row["expected_high"], None])
        y_lines.extend([row["symbol"], row["symbol"], None])
    fig.add_trace(
        go.Scatter(
            x=x_lines,
            y=y_lines,
            mode="lines",
            line=dict(color="#7f7f7f", width=8),
            name="7D IV expected range",
            hoverinfo="skip",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=df["spot"],
            y=df["symbol"],
            mode="markers",
            marker=dict(color="black", size=9, symbol="circle"),
            name="Spot",
            customdata=np.stack([df["expected_move_pct"], df["expected_move_abs"]], axis=-1),
            hovertemplate="%{y}<br>Spot=%{x:.2f}<br>Expected move=±%{customdata[0]:.2%} / ±%{customdata[1]:.2f}<extra></extra>",
        )
    )
    level_specs = [
        ("gamma_flip", "Gamma Flip", "#9467bd", "diamond"),
        ("call_wall", "Call Wall", "#2ca02c", "triangle-up"),
        ("put_wall", "Put Wall", "#d62728", "triangle-down"),
    ]
    for col, label, color, symbol in level_specs:
        fig.add_trace(
            go.Scatter(
                x=df[col],
                y=df["symbol"],
                mode="markers",
                marker=dict(color=color, size=10, symbol=symbol),
                name=label,
                hovertemplate=f"%{{y}}<br>{label}=%{{x:.2f}}<extra></extra>",
            )
        )

    fig.update_layout(
        title=f"IV Expected Move vs Key Levels ({IV_EXPECTED_MOVE_DAYS} calendar days, {IV_EXPECTED_MOVE_Z:.1f}σ)",
        xaxis_title="Price level",
        yaxis_title="Sector ETF",
        height=640,
        legend_orientation="h",
        legend_y=1.08,
    )
    return fig


def plot_iv_overlay_heatmap(iv_df: pd.DataFrame):
    if iv_df is None or iv_df.empty:
        fig = go.Figure()
        fig.update_layout(title="IV Overlay Heatmap: no data")
        return fig

    cols = [
        "expected_move_pct",
        "atm_iv",
        "iv_risk_score",
        "iv_adjusted_score",
        "gamma_flip_inside_expected_move",
        "call_wall_inside_expected_move",
        "put_wall_inside_expected_move",
    ]
    df = iv_df.sort_values("iv_adjusted_score", ascending=False).set_index("symbol")
    z = pd.DataFrame(index=df.index)
    for col in cols:
        if df[col].dtype == bool:
            z[col] = df[col].astype(int)
        else:
            z[col] = _zscore(pd.to_numeric(df[col], errors="coerce")).fillna(0)

    text = df[cols].copy()
    for col in ["expected_move_pct", "atm_iv"]:
        text[col] = text[col].map(lambda x: f"{x:.1%}" if pd.notna(x) else "")
    for col in ["iv_risk_score", "iv_adjusted_score"]:
        text[col] = text[col].map(lambda x: f"{x:.2f}" if pd.notna(x) else "")
    for col in ["gamma_flip_inside_expected_move", "call_wall_inside_expected_move", "put_wall_inside_expected_move"]:
        text[col] = text[col].map(lambda x: "Y" if bool(x) else "N")

    fig = go.Figure(
        data=go.Heatmap(
            z=z.values,
            x=cols,
            y=z.index,
            text=text.values,
            texttemplate="%{text}",
            colorscale="RdBu",
            zmid=0,
            colorbar_title="z / flag",
        )
    )
    fig.update_layout(title="IV Overlay Heatmap", height=540)
    return fig


def plot_vanna_charm_exposure_heatmap(second_order_df: pd.DataFrame):
    if second_order_df is None or second_order_df.empty:
        fig = go.Figure()
        fig.update_layout(title="Vanna/Charm Exposure Heatmap: no data")
        return fig

    cols = [
        "net_vanna_exposure",
        "abs_vanna_exposure",
        "near_5pct_vanna_exposure",
        "short_7d_vanna_exposure",
        "net_charm_exposure",
        "abs_charm_exposure",
        "near_5pct_charm_exposure",
        "short_7d_charm_exposure",
        "vanna_abs_concentration_ratio",
        "charm_abs_concentration_ratio",
        "high_vanna_risk",
        "high_charm_risk",
    ]
    available = [c for c in cols if c in second_order_df.columns]
    df = second_order_df.copy().set_index("symbol")
    z = pd.DataFrame(index=df.index)
    for col in available:
        if df[col].dtype == bool:
            z[col] = df[col].astype(int)
        else:
            z[col] = _zscore(pd.to_numeric(df[col], errors="coerce")).fillna(0)

    text = df[available].copy()
    for col in [c for c in available if "exposure" in c]:
        text[col] = pd.to_numeric(text[col], errors="coerce").map(lambda x: f"{x/1e6:.1f}mm" if pd.notna(x) else "")
    for col in ["vanna_abs_concentration_ratio", "charm_abs_concentration_ratio"]:
        if col in text:
            text[col] = pd.to_numeric(text[col], errors="coerce").map(lambda x: f"{x:.0%}" if pd.notna(x) else "")
    for col in ["high_vanna_risk", "high_charm_risk"]:
        if col in text:
            text[col] = text[col].map(lambda x: "Y" if bool(x) else "N")

    fig = go.Figure(
        data=go.Heatmap(
            z=z[available].values,
            x=available,
            y=z.index,
            text=text[available].values,
            texttemplate="%{text}",
            colorscale="RdBu",
            zmid=0,
            colorbar_title="z / flag",
        )
    )
    fig.update_layout(title="Vanna/Charm Exposure Heatmap", height=620)
    return fig


def plot_vanna_charm_by_strike(chains_by_symbol: dict[str, pd.DataFrame], summary_df: pd.DataFrame, range_mult=0.20):
    fig = go.Figure()
    buttons = []
    symbols = [s for s in SECTOR_ETFS if s in chains_by_symbol and not chains_by_symbol[s].empty]
    if not symbols:
        fig.update_layout(title="Vanna/Charm by Strike: no option data")
        return fig

    first_symbol = symbols[0]
    shapes_by_symbol = {}
    trace_idx = 0
    for symbol in symbols:
        chain = add_gamma_and_gex(chains_by_symbol[symbol])
        row = summary_df[summary_df["symbol"].eq(symbol)].iloc[0]
        spot = row["spot"]
        view = chain[(chain["strike"] >= spot * (1 - range_mult)) & (chain["strike"] <= spot * (1 + range_mult))]
        agg = (
            view.groupby("strike", as_index=False)[["signed_vanna_exposure", "signed_charm_exposure"]].sum()
            if not view.empty
            else pd.DataFrame(columns=["strike", "signed_vanna_exposure", "signed_charm_exposure"])
        )
        visible = symbol == first_symbol
        fig.add_bar(x=agg["strike"], y=agg["signed_vanna_exposure"], name=f"{symbol} Vanna", marker_color="#9467bd", visible=visible)
        fig.add_bar(x=agg["strike"], y=agg["signed_charm_exposure"], name=f"{symbol} Charm", marker_color="#ff7f0e", visible=visible)
        shapes_by_symbol[symbol] = _wall_shapes(row)

        visibility = [False] * (2 * len(symbols))
        visibility[trace_idx] = True
        visibility[trace_idx + 1] = True
        buttons.append(
            {
                "label": symbol,
                "method": "update",
                "args": [
                    {"visible": visibility},
                    {"title": f"Vanna/Charm by Strike: {symbol}", "shapes": shapes_by_symbol[symbol]},
                ],
            }
        )
        trace_idx += 2

    fig.update_layout(
        title=f"Vanna/Charm by Strike: {first_symbol}",
        barmode="relative",
        xaxis_title="Strike",
        yaxis_title="Signed Exposure",
        shapes=shapes_by_symbol[first_symbol],
        updatemenus=[{"buttons": buttons, "direction": "down", "x": 0.0, "y": 1.15}],
        height=600,
    )
    fig.add_hline(y=0, line_width=1, line_color="black")
    return fig


## 10. Save Outputs


In [11]:
def save_outputs(
    chains_by_symbol: dict[str, pd.DataFrame],
    gex_summary_df: pd.DataFrame,
    sector_scores_df: pd.DataFrame,
    proposed_portfolio_df: pd.DataFrame,
    gex_decision_table_df: pd.DataFrame,
    figures: dict[str, go.Figure],
    iv_expected_moves_df: pd.DataFrame | None = None,
    second_order_greeks_df: pd.DataFrame | None = None,
    weekly_attribution_df: pd.DataFrame | None = None,
    weekly_attribution_summary: dict | None = None,
    as_of: str = AS_OF,
) -> dict[str, str]:
    raw_dir = PROJECT_ROOT / "data" / "raw" / "options" / as_of
    processed_dir = PROJECT_ROOT / "data" / "processed" / as_of
    dashboard_dir = PROJECT_ROOT / "outputs" / "dashboard"
    table_dir = PROJECT_ROOT / "outputs" / "tables"
    for path in [raw_dir, processed_dir, dashboard_dir, table_dir]:
        path.mkdir(parents=True, exist_ok=True)

    paths = {}
    for symbol, chain in chains_by_symbol.items():
        if chain is None or chain.empty:
            continue
        path = raw_dir / f"{symbol}_option_chain.parquet"
        chain.to_parquet(path, index=False)
        paths[f"raw_options_{symbol}"] = str(path)

    summary_path = processed_dir / "sector_gex_summary.parquet"
    scores_path = processed_dir / "sector_scores.parquet"
    portfolio_path = processed_dir / "proposed_portfolio.parquet"
    gex_summary_df.to_parquet(summary_path, index=False)
    sector_scores_df.to_parquet(scores_path, index=False)
    proposed_portfolio_df.to_parquet(portfolio_path, index=False)
    paths["sector_gex_summary"] = str(summary_path)
    paths["sector_scores"] = str(scores_path)
    paths["proposed_portfolio_parquet"] = str(portfolio_path)

    portfolio_csv_path = table_dir / "proposed_portfolio.csv"
    decision_csv_path = table_dir / "gex_decision_table.csv"
    proposed_portfolio_df.to_csv(portfolio_csv_path, index=False)
    gex_decision_table_df.to_csv(decision_csv_path, index=False)
    paths["proposed_portfolio_csv"] = str(portfolio_csv_path)
    paths["gex_decision_table_csv"] = str(decision_csv_path)

    if iv_expected_moves_df is not None and not iv_expected_moves_df.empty:
        iv_parquet_path = processed_dir / "sector_iv_expected_moves.parquet"
        iv_csv_path = table_dir / "iv_expected_move_table.csv"
        iv_expected_moves_df.to_parquet(iv_parquet_path, index=False)
        iv_expected_moves_df.to_csv(iv_csv_path, index=False)
        paths["sector_iv_expected_moves"] = str(iv_parquet_path)
        paths["iv_expected_move_table_csv"] = str(iv_csv_path)

    if second_order_greeks_df is not None and not second_order_greeks_df.empty:
        second_order_parquet_path = processed_dir / "sector_second_order_greeks.parquet"
        second_order_csv_path = table_dir / "second_order_greeks_table.csv"
        second_order_greeks_df.to_parquet(second_order_parquet_path, index=False)
        second_order_greeks_df.to_csv(second_order_csv_path, index=False)
        paths["sector_second_order_greeks"] = str(second_order_parquet_path)
        paths["second_order_greeks_table_csv"] = str(second_order_csv_path)

    if weekly_attribution_df is not None and not weekly_attribution_df.empty:
        attribution_parquet_path = processed_dir / "weekly_attribution.parquet"
        attribution_csv_path = table_dir / "weekly_attribution.csv"
        weekly_attribution_df.to_parquet(attribution_parquet_path, index=False)
        weekly_attribution_df.to_csv(attribution_csv_path, index=False)
        paths["weekly_attribution_parquet"] = str(attribution_parquet_path)
        paths["weekly_attribution_csv"] = str(attribution_csv_path)

    dashboard_path = dashboard_dir / "sector_gamma_dashboard.html"
    kpi_cols = [
        "symbol",
        "sector_name",
        "proposed_weight",
        "active_weight",
        "total_score",
        "momentum_score",
        "gamma_score",
        "option_demand_score",
        "risk_control_score",
        "overweight_allowed",
        "high_vanna_risk",
        "high_charm_risk",
        "gex_regime_comment",
        "vanna_charm_risk_comment",
    ]
    html_parts = [
        "<html><head><meta charset='utf-8'><title>Sector Gamma Dashboard</title></head><body>",
        f"<h1>Sector Gamma Dashboard</h1><p>As of {as_of}</p>",
        "<h2>KPI Table</h2>",
        proposed_portfolio_df[[c for c in kpi_cols if c in proposed_portfolio_df.columns]].to_html(index=False),
        "<h2>GEX Decision Table</h2>",
        gex_decision_table_df.to_html(index=False),
    ]
    if iv_expected_moves_df is not None and not iv_expected_moves_df.empty:
        html_parts.extend([
            "<h2>IV Expected Move Overlay</h2>",
            iv_expected_moves_df.to_html(index=False),
        ])
    if second_order_greeks_df is not None and not second_order_greeks_df.empty:
        html_parts.extend([
            "<h2>Vanna/Charm Second-Order Greeks</h2>",
            second_order_greeks_df.to_html(index=False),
        ])
    if weekly_attribution_df is not None and not weekly_attribution_df.empty:
        html_parts.append("<h2>Previous Week Attribution</h2>")
        if weekly_attribution_summary:
            html_parts.append(pd.DataFrame([weekly_attribution_summary]).to_html(index=False))
        html_parts.append(weekly_attribution_df.to_html(index=False))
    include_js = True
    for title, fig in figures.items():
        html_parts.append(f"<h2>{title}</h2>")
        html_parts.append(pio.to_html(fig, full_html=False, include_plotlyjs=include_js))
        include_js = False
    html_parts.append("</body></html>")
    dashboard_path.write_text("\n".join(html_parts), encoding="utf-8")
    paths["dashboard_html"] = str(dashboard_path)
    return paths


## 11. Validation


In [12]:
def _validation_row(category: str, check: str, ok: bool, detail: str = "") -> dict:
    return {"category": category, "check": check, "status": "PASS" if ok else "WARN", "detail": detail}


def validate_outputs(
    chains_by_symbol: dict[str, pd.DataFrame],
    gex_summary_df: pd.DataFrame,
    proposed_portfolio_df: pd.DataFrame,
    figures: dict[str, go.Figure],
    output_paths: dict[str, str] | None = None,
    iv_expected_moves_df: pd.DataFrame | None = None,
    second_order_greeks_df: pd.DataFrame | None = None,
    weekly_attribution_df: pd.DataFrame | None = None,
    weekly_attribution_summary: dict | None = None,
) -> pd.DataFrame:
    rows = []

    for symbol in SECTOR_ETFS:
        chain = chains_by_symbol.get(symbol, pd.DataFrame()) if chains_by_symbol else pd.DataFrame()
        rows.append(_validation_row("data", f"{symbol} option chain non-empty", not chain.empty, f"rows={len(chain):,}"))
        if not chain.empty:
            rows.append(_validation_row("data", f"{symbol} openInterest sum non-zero", chain["openInterest"].fillna(0).sum() > 0, str(chain["openInterest"].fillna(0).sum())))
            valid_iv_ratio = chain["impliedVolatility"].between(0, IV_MAX, inclusive="neither").mean()
            rows.append(_validation_row("data", f"{symbol} impliedVolatility mostly valid", valid_iv_ratio > 0.5, f"valid_ratio={valid_iv_ratio:.1%}"))
            if "gamma" in chain:
                rows.append(_validation_row("data", f"{symbol} gamma not all NaN", not chain["gamma"].isna().all(), ""))
            for col in ["vanna_1vol", "charm_1d"]:
                if col in chain:
                    rows.append(_validation_row("data", f"{symbol} {col} not all NaN", not chain[col].isna().all(), ""))
            for col in ["signed_vanna_exposure", "signed_charm_exposure"]:
                if col in chain:
                    finite_ratio = np.isfinite(pd.to_numeric(chain[col], errors="coerce")).mean()
                    rows.append(_validation_row("data", f"{symbol} {col} finite", finite_ratio > 0.5, f"finite_ratio={finite_ratio:.1%}"))

    if gex_summary_df is not None and not gex_summary_df.empty:
        rows.append(_validation_row("data", "abs_gex non-zero for at least one ETF", (gex_summary_df["abs_gex"].fillna(0) > 0).any(), ""))

    if second_order_greeks_df is not None and not second_order_greeks_df.empty:
        rows.append(_validation_row("second_order", "second_order_greeks has all 11 sectors", len(set(second_order_greeks_df["symbol"])) == len(SECTOR_ETFS), ""))
        rows.append(_validation_row("second_order", "abs_vanna_exposure non-zero for at least one ETF", (second_order_greeks_df["abs_vanna_exposure"].fillna(0) > 0).any(), ""))
        rows.append(_validation_row("second_order", "abs_charm_exposure non-zero for at least one ETF", (second_order_greeks_df["abs_charm_exposure"].fillna(0) > 0).any(), ""))
        for col in ["high_vanna_risk", "high_charm_risk"]:
            rows.append(_validation_row("second_order", f"{col} is boolean", second_order_greeks_df[col].map(lambda x: isinstance(x, (bool, np.bool_))).all(), ""))

    if weekly_attribution_df is not None and not weekly_attribution_df.empty:
        symbols = set(weekly_attribution_df["symbol"])
        rows.append(_validation_row("attribution", "weekly attribution has all 11 sectors", symbols == set(SECTOR_ETFS), f"available={sorted(symbols)}"))
        dates_ok = (
            weekly_attribution_df["start_date"].notna().all()
            and weekly_attribution_df["end_date"].notna().all()
            and (pd.to_datetime(weekly_attribution_df["start_date"]) < pd.to_datetime(weekly_attribution_df["end_date"])).all()
        )
        rows.append(_validation_row("attribution", "weekly attribution dates are valid", dates_ok, ""))
        numeric_cols = ["realized_return", "benchmark_weight", "active_weight", "proposed_weight", "benchmark_contribution", "active_contribution", "proposed_contribution"]
        finite_ok = np.isfinite(weekly_attribution_df[numeric_cols].apply(pd.to_numeric, errors="coerce")).all().all()
        rows.append(_validation_row("attribution", "weekly attribution values are finite", finite_ok, ""))
        row_identity = np.allclose(
            weekly_attribution_df["proposed_contribution"] - weekly_attribution_df["benchmark_contribution"],
            weekly_attribution_df["active_contribution"],
            atol=1e-12,
        )
        rows.append(_validation_row("attribution", "row contribution identity holds", row_identity, ""))
        summary_ok = bool(weekly_attribution_summary) and np.isclose(
            weekly_attribution_summary.get("proposed_return", np.nan) - weekly_attribution_summary.get("benchmark_return", np.nan),
            weekly_attribution_summary.get("active_return", np.nan),
            atol=1e-12,
        )
        rows.append(_validation_row("attribution", "portfolio active return identity holds", summary_ok, str(weekly_attribution_summary)))
    else:
        rows.append(_validation_row("attribution", "weekly attribution is available", False, "No prior snapshot or completed price window."))

    if proposed_portfolio_df is not None and not proposed_portfolio_df.empty:
        rows.append(
            _validation_row(
                "portfolio",
                "proposed_weight sums to 1.0",
                abs(proposed_portfolio_df["proposed_weight"].sum() - 1.0) < 1e-8,
                f"sum={proposed_portfolio_df['proposed_weight'].sum():.10f}",
            )
        )
        rows.append(
            _validation_row(
                "portfolio",
                "active_weight sums near 0",
                abs(proposed_portfolio_df["active_weight"].sum()) < 1e-8,
                f"sum={proposed_portfolio_df['active_weight'].sum():.10f}",
            )
        )
        rows.append(
            _validation_row(
                "portfolio",
                "proposed_weight has no negative values",
                (proposed_portfolio_df["proposed_weight"] >= -1e-12).all(),
                "",
            )
        )
        missing_bm = sorted(set(SECTOR_ETFS) - set(proposed_portfolio_df.dropna(subset=["benchmark_weight"])["symbol"]))
        rows.append(_validation_row("portfolio", "benchmark_weight exists for all sectors", len(missing_bm) == 0, f"missing={missing_bm}"))

    expected_figs = {
        "allocation",
        "active_tilt",
        "signal_heatmap",
        "gex_indicator_heatmap",
        "gex_by_strike",
        "gamma_flip_curve",
        "price_with_walls",
        "iv_expected_move_bands",
        "iv_overlay_heatmap",
        "Vanna/Charm Exposure Heatmap",
        "Vanna/Charm by Strike",
    }
    fig_keys = set(figures or {})
    rows.append(_validation_row("visualization", "all expected Plotly figures built", expected_figs.issubset(fig_keys), f"built={sorted(fig_keys)}"))
    dropdown_symbols = sorted([s for s in SECTOR_ETFS if chains_by_symbol and s in chains_by_symbol and not chains_by_symbol[s].empty])
    rows.append(_validation_row("visualization", "dropdown includes all 11 sectors with option data", len(dropdown_symbols) == 11, f"available={dropdown_symbols}"))

    if iv_expected_moves_df is not None and not iv_expected_moves_df.empty:
        rows.append(_validation_row("iv", "atm_iv exists for all sectors", iv_expected_moves_df["atm_iv"].notna().all(), ""))
        rows.append(_validation_row("iv", "expected_move_pct positive", (iv_expected_moves_df["expected_move_pct"] > 0).all(), ""))
        rows.append(
            _validation_row(
                "iv",
                "expected_low < spot < expected_high",
                ((iv_expected_moves_df["expected_low"] < iv_expected_moves_df["spot"]) & (iv_expected_moves_df["spot"] < iv_expected_moves_df["expected_high"])).all(),
                "",
            )
        )
        for col in ["call_wall_inside_expected_move", "put_wall_inside_expected_move", "gamma_flip_inside_expected_move"]:
            rows.append(_validation_row("iv", f"{col} is boolean", iv_expected_moves_df[col].map(lambda x: isinstance(x, (bool, np.bool_))).all(), ""))

    if output_paths:
        dashboard_path = output_paths.get("dashboard_html")
        rows.append(_validation_row("visualization", "HTML dashboard saved", bool(dashboard_path and Path(dashboard_path).exists()), str(dashboard_path)))
        iv_parquet_path = output_paths.get("sector_iv_expected_moves")
        iv_csv_path = output_paths.get("iv_expected_move_table_csv")
        rows.append(_validation_row("iv", "sector_iv_expected_moves parquet saved", bool(iv_parquet_path and Path(iv_parquet_path).exists()), str(iv_parquet_path)))
        rows.append(_validation_row("iv", "iv_expected_move_table csv saved", bool(iv_csv_path and Path(iv_csv_path).exists()), str(iv_csv_path)))
        second_order_parquet_path = output_paths.get("sector_second_order_greeks")
        second_order_csv_path = output_paths.get("second_order_greeks_table_csv")
        rows.append(_validation_row("second_order", "sector_second_order_greeks parquet saved", bool(second_order_parquet_path and Path(second_order_parquet_path).exists()), str(second_order_parquet_path)))
        rows.append(_validation_row("second_order", "second_order_greeks_table csv saved", bool(second_order_csv_path and Path(second_order_csv_path).exists()), str(second_order_csv_path)))
        attribution_parquet_path = output_paths.get("weekly_attribution_parquet")
        attribution_csv_path = output_paths.get("weekly_attribution_csv")
        rows.append(_validation_row("attribution", "weekly attribution parquet saved", bool(attribution_parquet_path and Path(attribution_parquet_path).exists()), str(attribution_parquet_path)))
        rows.append(_validation_row("attribution", "weekly attribution csv saved", bool(attribution_csv_path and Path(attribution_csv_path).exists()), str(attribution_csv_path)))

    return pd.DataFrame(rows)


## 実行設定

`MAX_EXPIRIES = None` は全満期を取得する。初回の確認を軽くしたい場合は `MAX_EXPIRIES = 3` などに変更する。


In [13]:
RUN_FETCH = True
PRICE_SYMBOLS = SECTOR_ETFS + ["SPY"]

print(f"RUN_FETCH={RUN_FETCH}")
print(f"PRICE_PERIOD={PRICE_PERIOD}")
print(f"MAX_EXPIRIES={MAX_EXPIRIES}")
print(f"IV_EXPECTED_MOVE_DAYS={IV_EXPECTED_MOVE_DAYS}")
print(f"IV_EXPECTED_MOVE_Z={IV_EXPECTED_MOVE_Z}")


RUN_FETCH=True
PRICE_PERIOD=1y
MAX_EXPIRIES=None
IV_EXPECTED_MOVE_DAYS=7
IV_EXPECTED_MOVE_Z=1.0


## Price Data 実行


In [14]:
if RUN_FETCH:
    price_df = get_price_data(PRICE_SYMBOLS, period=PRICE_PERIOD)
    price_signals = compute_price_signals(price_df)
else:
    price_df = pd.DataFrame(columns=["date", "symbol", "close"])
    price_signals = pd.DataFrame()

display(price_signals)


,symbol,price,ret20,ret60,rel_ret20_vs_spy,rel_ret60_vs_spy,ma20,ma50,dd_60,above_20ma,above_50ma,drawdown_gt_7pct,momentum_score
0,XLK,176.199997,-0.078510,0.115884,-0.075148,0.066760,182.914000,182.850815,-0.109984,False,False,True,-1.00
1,XLY,114.839996,-0.017849,-0.032466,-0.014488,-0.081590,116.122500,117.002810,-0.057276,False,False,False,-2.00
2,XLP,85.070000,0.028335,0.043238,0.031696,-0.005886,84.147000,83.833260,-0.008624,True,True,False,1.75
3,XLE,58.070000,0.087759,0.034467,0.091120,-0.014656,54.948500,56.297506,-0.045704,True,True,False,1.75
4,XLF,56.099998,0.050896,0.078271,0.054258,0.029147,55.119000,53.131115,-0.011454,True,True,False,2.50
5,XLV,160.300003,0.077691,0.099925,0.081052,0.050801,159.356500,152.793662,-0.025176,True,True,False,2.50
6,XLI,178.550003,-0.010617,0.046476,-0.007256,-0.002647,181.555999,176.854156,-0.037778,False,True,False,-0.50
7,XLC,111.135002,0.018037,-0.054767,0.021398,-0.103890,109.523749,112.170059,-0.050821,True,False,False,0.25
8,XLB,50.134998,-0.028712,-0.029086,-0.025350,-0.078210,50.984249,50.969175,-0.045477,False,False,False,-2.00
9,XLRE,45.320000,0.042342,0.051936,0.045704,0.002812,44.638000,44.219970,-0.003080,True,True,False,2.50


## 前週パフォーマンス検証

直前の週次スナップショットで提案した配分を、見通し対象週の配当調整済みETF終値で評価する。Benchmark / Proposed / Active の収益率とセクター別寄与度を表示する。


In [15]:
weekly_attribution, weekly_attribution_summary = compute_previous_week_attribution(
    price_df=price_df,
    processed_root=PROJECT_ROOT / "data" / "processed",
    as_of=AS_OF,
)

if weekly_attribution.empty:
    display(HTML("<p><strong>Weekly attribution unavailable.</strong> See warnings above.</p>"))
else:
    display(pd.DataFrame([weekly_attribution_summary]))
    display(weekly_attribution)


,prior_snapshot,start_date,end_date,benchmark_return,proposed_return,active_return,complete
0,20260712,2026-07-10,2026-07-17,-0.017593,-0.018356,-0.000763,True


,symbol,sector_name,prior_decision,prior_snapshot,start_date,end_date,start_close,end_close,realized_return,benchmark_weight,active_weight,proposed_weight,benchmark_contribution,active_contribution,proposed_contribution,prior_total_score,prior_iv_adjusted_score,prior_overweight_allowed
0,XLF,Financials,Overweight,20260712,2026-07-10,2026-07-17,55.709999,56.259998,0.009873,0.125874,2.000000e-02,0.145874,0.001243,1.974508e-04,0.001440,1.325,1.225,True
1,XLC,Communication Services,Underweight,20260712,2026-07-10,2026-07-17,111.639999,110.650002,-0.008868,0.102897,-1.166667e-02,0.091230,-0.000912,1.034573e-04,-0.000809,-0.975,-1.075,False
2,XLRE,Real Estate,Neutral,20260712,2026-07-10,2026-07-17,44.450001,45.419998,0.021822,0.019980,3.469447e-18,0.019980,0.000436,7.571101e-20,0.000436,0.350,0.250,False
3,XLV,Health Care,Neutral,20260712,2026-07-10,2026-07-17,160.839996,161.089996,0.001554,0.094905,1.387779e-17,0.094905,0.000148,2.157080e-20,0.000148,0.975,0.775,False
4,XLU,Utilities,Neutral,20260712,2026-07-10,2026-07-17,45.410000,45.169998,-0.005285,0.024975,3.469447e-18,0.024975,-0.000132,-1.833678e-20,-0.000132,0.025,-0.325,False
5,XLB,Materials,Neutral,20260712,2026-07-10,2026-07-17,50.889999,50.529999,-0.007074,0.020979,3.469447e-18,0.020979,-0.000148,-2.454319e-20,-0.000148,-0.525,-0.525,False
6,XLI,Industrials,Neutral,20260712,2026-07-10,2026-07-17,181.919998,179.410004,-0.013797,0.089910,1.387779e-17,0.089910,-0.001241,-1.914752e-19,-0.001241,0.200,0.100,False
7,XLK,Information Technology,Neutral,20260712,2026-07-10,2026-07-17,185.779999,175.589996,-0.054850,0.328671,5.551115e-17,0.328671,-0.018028,-3.044778e-18,-0.018028,1.350,1.050,False
8,XLP,Consumer Staples,Underweight,20260712,2026-07-10,2026-07-17,84.120003,85.190002,0.012720,0.052947,-7.777778e-03,0.045169,0.000673,-9.893271e-05,0.000575,-0.725,-0.975,False
9,XLY,Consumer Discretionary,Overweight,20260712,2026-07-10,2026-07-17,117.239998,115.440002,-0.015353,0.098901,1.500000e-02,0.113901,-0.001518,-2.302962e-04,-0.001749,0.275,0.175,True


## Option Chain / GEX 実行


In [16]:
if RUN_FETCH:
    option_chains = fetch_all_option_chains(SECTOR_ETFS, max_expiries=MAX_EXPIRIES)
    option_chains = {symbol: add_gamma_and_gex(chain) for symbol, chain in option_chains.items()}
else:
    option_chains = {}

summary_records = []
second_order_records = []
gex_grids = {}
for symbol in SECTOR_ETFS:
    chain = option_chains.get(symbol, pd.DataFrame())
    summary, grid = summarize_option_gamma_extended(chain)
    summary["symbol"] = summary.get("symbol") or symbol
    summary_records.append(summary)

    second_order_summary = summarize_second_order_greeks(chain)
    second_order_summary["symbol"] = second_order_summary.get("symbol") or symbol
    second_order_records.append(second_order_summary)

    if not grid.empty:
        grid = grid.copy()
        grid["symbol"] = symbol
    gex_grids[symbol] = grid

gex_summary = pd.DataFrame(summary_records)
second_order_greeks = add_vanna_charm_flags(pd.DataFrame(second_order_records))
display(gex_summary)
display(HTML("<h3>Vanna/Charm Second-Order Greeks</h3>"))
display(second_order_greeks)


Fetching option chain: XLK


  rows: 2,180
Fetching option chain: XLY


  rows: 951
Fetching option chain: XLP


  rows: 644
Fetching option chain: XLE


  rows: 1,649
Fetching option chain: XLF


...

  rows: 620
Fetching option chain: XLRE


  rows: 117
Fetching option chain: XLU


  rows: 953


,symbol,spot,net_gex,abs_gex,gamma_flip,spot_to_flip_pct,call_wall,put_wall,spot_to_call_wall_pct,spot_to_put_wall_pct,near_2pct_gex,near_2pct_abs_gex,near_5pct_gex,near_5pct_abs_gex,short_7d_gex,short_7d_abs_gex,short_14d_gex,short_14d_abs_gex,near_5pct_abs_gex_ratio,short_7d_abs_gex_ratio,short_14d_abs_gex_ratio,call_oi,put_oi,call_volume,put_volume,put_call_oi_ratio,put_call_volume_ratio,atm_iv
0,XLK,176.199997,-1.399264e+07,1.436385e+08,180.693901,0.025505,200.0,170.0,0.135074,-0.035187,-1.059216e+07,2.294229e+07,-1.680857e+07,6.309551e+07,-4.280914e+06,2.404355e+07,-6.330462e+06,3.376645e+07,0.439266,0.167389,0.235079,272163.0,424216.0,12025.0,38066.0,1.558684,3.165572,0.351691
1,XLY,114.849998,-7.349819e+07,1.291945e+08,137.819998,0.200000,120.0,107.5,0.044841,-0.063997,-1.456992e+07,2.195361e+07,-3.768590e+07,6.328257e+07,1.873483e+05,1.090196e+06,1.641009e+06,3.183539e+06,0.489824,0.008438,0.024641,112890.0,416615.0,9761.0,136134.0,3.690451,13.946727,0.255501
2,XLP,85.070000,-2.610883e+07,1.036807e+08,88.519105,0.040544,85.0,78.0,-0.000823,-0.083108,-1.086072e+06,4.707337e+07,-5.176122e+06,7.054191e+07,7.724088e+05,1.407201e+07,-1.534779e+06,2.251874e+07,0.680376,0.135724,0.217193,82097.0,264345.0,6721.0,14147.0,3.219911,2.104895,0.161629
3,XLE,58.075001,4.433447e+07,3.722114e+08,56.349960,-0.029704,60.0,55.0,0.033147,-0.052949,2.880297e+07,8.405278e+07,7.306102e+07,1.577905e+08,1.895412e+07,2.514700e+07,3.222771e+07,4.054917e+07,0.423927,0.067561,0.108941,1761194.0,2552053.0,30665.0,66677.0,1.449047,2.174368,0.288276
4,XLF,56.115002,1.378159e+08,6.340553e+08,53.919449,-0.039126,56.0,50.0,-0.002049,-0.108973,1.234163e+08,2.125486e+08,1.671356e+08,3.176333e+08,1.576033e+07,3.118879e+07,2.973552e+07,6.063562e+07,0.500955,0.049189,0.095631,2320538.0,3129442.0,22993.0,91919.0,1.348585,3.997695,0.164559
5,XLV,160.279999,4.565183e+07,1.888704e+08,151.744240,-0.053255,165.0,155.0,0.029448,-0.032942,2.723282e+06,5.410207e+07,2.247225e+07,1.154142e+08,6.077347e+06,2.059804e+07,4.927072e+06,3.013176e+07,0.611076,0.109059,0.159537,228687.0,315320.0,8735.0,10471.0,1.378828,1.198741,0.184151
6,XLI,178.570007,-9.872173e+07,1.872596e+08,193.710579,0.084788,180.0,165.0,0.008008,-0.075993,-2.698438e+07,5.332361e+07,-3.940614e+07,8.798294e+07,-4.056342e+06,9.086092e+06,-2.625241e+07,3.572460e+07,0.469845,0.048521,0.190776,100407.0,341529.0,8002.0,86378.0,3.401446,10.794551,0.257576
7,XLC,111.180000,-4.779397e+07,6.595149e+07,127.112991,0.143308,115.0,104.0,0.034359,-0.064580,-2.435956e+06,7.263756e+06,-4.178097e+06,1.438559e+07,2.332793e+05,1.232505e+06,2.704511e+04,2.112573e+06,0.218124,0.018688,0.032032,29373.0,211932.0,2807.0,4384.0,7.215198,1.561810,0.215157
8,XLB,50.165001,9.937728e+06,5.605245e+07,48.706030,-0.029083,55.0,47.5,0.096382,-0.053125,-1.937379e+06,9.477410e+06,-2.998290e+06,1.516829e+07,-6.302927e+05,4.231539e+06,-1.164527e+06,6.677665e+06,0.270609,0.075492,0.119132,302363.0,191161.0,5180.0,125223.0,0.632224,24.174324,0.207283
9,XLRE,45.320000,1.691641e+06,3.377738e+06,41.658543,-0.080791,46.0,35.0,0.015004,-0.227714,1.365497e+06,1.646564e+06,1.573375e+06,2.174781e+06,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.643857,0.000000,0.000000,14782.0,13906.0,327.0,1943.0,0.940739,5.941896,0.242805


,symbol,spot,net_vanna_exposure,abs_vanna_exposure,net_charm_exposure,abs_charm_exposure,near_5pct_vanna_exposure,near_5pct_charm_exposure,short_7d_vanna_exposure,short_7d_charm_exposure,near_5pct_abs_vanna_exposure,near_5pct_abs_charm_exposure,short_7d_abs_vanna_exposure,short_7d_abs_charm_exposure,vanna_near_5pct_abs_ratio,vanna_short_7d_abs_ratio,charm_near_5pct_abs_ratio,charm_short_7d_abs_ratio,vanna_abs_concentration_ratio,charm_abs_concentration_ratio,high_vanna_risk,high_charm_risk,vanna_charm_risk_comment
0,XLK,176.199997,2.132852e+07,3.519118e+07,-1.104138e+07,1.929094e+07,1.413941e+06,-3.674844e+06,8.876421e+05,-5.048284e+06,3.898277e+06,7.858421e+06,1.507167e+06,8.830629e+06,0.110774,0.042828,0.407363,0.457760,0.110774,0.457760,False,True,Charm集中が高い。時間経過に伴うdealer delta調整に注意（net Charm 負）。
1,XLY,114.849998,2.592458e+07,3.095357e+07,-4.318104e+06,5.659126e+06,5.217037e+06,-1.076153e+06,6.038809e+04,-3.192556e+05,7.426447e+06,1.945678e+06,7.483670e+04,3.949949e+05,0.239922,0.002418,0.343812,0.069798,0.239922,0.343812,False,False,Vanna/Charmの集中は相対的に限定的。Gamma/GEX水準を主軸に判断。
2,XLP,85.070000,2.071558e+07,2.429952e+07,-3.083413e+06,5.777267e+06,4.761342e+06,-1.237341e+06,7.829246e+04,-9.657430e+04,7.415095e+06,3.835980e+06,7.717312e+05,2.097072e+06,0.305154,0.031759,0.663978,0.362987,0.305154,0.663978,False,True,Charm集中が高い。時間経過に伴うdealer delta調整に注意（net Charm 負）。
3,XLE,58.075001,7.218050e+07,1.094850e+08,-1.331902e+07,2.503195e+07,4.311975e+06,-2.819680e+06,4.335660e+05,-1.796887e+06,1.087064e+07,8.925868e+06,1.298020e+06,5.898182e+06,0.099289,0.011856,0.356579,0.235626,0.099289,0.356579,False,True,Charm集中が高い。時間経過に伴うdealer delta調整に注意（net Charm 負）。
4,XLF,56.115002,1.325814e+08,2.068965e+08,-1.910081e+07,3.437624e+07,1.974851e+06,-4.575177e+06,1.415676e+06,-3.358517e+06,3.620736e+07,1.243953e+07,2.737988e+06,6.910232e+06,0.175002,0.013234,0.361864,0.201018,0.175002,0.361864,False,True,Charm集中が高い。時間経過に伴うdealer delta調整に注意（net Charm 負）。
5,XLV,160.279999,2.967028e+07,4.891772e+07,-7.819291e+06,1.137850e+07,6.283537e+06,-4.482307e+06,1.142777e+06,-2.755271e+06,1.375504e+07,6.930099e+06,1.603592e+06,4.462570e+06,0.281187,0.032781,0.609052,0.392193,0.281187,0.609052,True,True,Vanna/Charmの集中が高い。IV変化と1日時間経過でdealer delta調整が出...
6,XLI,178.570007,4.423134e+07,4.862018e+07,-1.055928e+07,1.313370e+07,4.864004e+06,-2.171818e+06,3.945274e+05,-1.492164e+06,7.525189e+06,4.443392e+06,6.259834e+05,2.420405e+06,0.154775,0.012875,0.338320,0.184290,0.154775,0.338320,False,False,Vanna/Charmの集中は相対的に限定的。Gamma/GEX水準を主軸に判断。
7,XLC,111.180000,1.971091e+07,2.134616e+07,-3.290343e+06,3.838832e+06,7.267156e+05,-1.896093e+05,3.363218e+04,-1.659075e+05,1.585107e+06,5.915956e+05,6.270641e+04,3.184843e+05,0.074257,0.002938,0.154108,0.082964,0.074257,0.154108,False,False,Vanna/Charmの集中は相対的に限定的。Gamma/GEX水準を主軸に判断。
8,XLB,50.165001,1.666799e+07,1.773501e+07,-4.340795e+06,4.935144e+06,6.549629e+05,-6.375190e+05,1.342727e+05,-5.404525e+05,1.192871e+06,1.159151e+06,2.299326e+05,8.496847e+05,0.067261,0.012965,0.234877,0.172170,0.067261,0.234877,False,False,Vanna/Charmの集中は相対的に限定的。Gamma/GEX水準を主軸に判断。
9,XLRE,45.320000,6.301223e+05,8.520552e+05,-9.990664e+04,1.314335e+05,1.014969e+05,-4.744527e+04,0.000000e+00,0.000000e+00,1.990430e+05,5.411680e+04,0.000000e+00,0.000000e+00,0.233603,0.000000,0.411743,0.000000,0.233603,0.411743,False,False,Vanna/Charmの集中は相対的に限定的。Gamma/GEX水準を主軸に判断。


## Scoring / Portfolio 実行


In [17]:
option_scores = compute_option_scores_extended(gex_summary)
sector_scores = compute_total_scores(price_signals, option_scores)
proposed_portfolio = construct_portfolio(sector_scores, BENCHMARK_WEIGHTS)
proposed_portfolio = compute_iv_overlay_scores(proposed_portfolio)
if second_order_greeks is not None and not second_order_greeks.empty:
    second_order_overlay_cols = [c for c in second_order_greeks.columns if c not in {"spot"}]
    proposed_portfolio = proposed_portfolio.merge(second_order_greeks[second_order_overlay_cols], on="symbol", how="left")
iv_expected_moves = make_iv_expected_move_table(proposed_portfolio)
gex_decision_table = make_gex_decision_table(proposed_portfolio)

kpi_cols = [
    "symbol",
    "sector_name",
    "proposed_weight",
    "active_weight",
    "total_score",
    "momentum_score",
    "gamma_score",
    "option_demand_score",
    "risk_control_score",
    "overweight_allowed",
    "high_vanna_risk",
    "high_charm_risk",
    "gex_regime_comment",
    "vanna_charm_risk_comment",
]
iv_cols = [
    "symbol",
    "sector_name",
    "atm_iv",
    "expected_move_pct",
    "expected_low",
    "spot",
    "expected_high",
    "gamma_flip_inside_expected_move",
    "call_wall_inside_expected_move",
    "put_wall_inside_expected_move",
    "iv_risk_score",
    "total_score",
    "iv_adjusted_score",
    "iv_weekly_comment",
]
display(proposed_portfolio[[c for c in kpi_cols if c in proposed_portfolio.columns]])
display(iv_expected_moves[iv_cols])
display(gex_decision_table)


,symbol,sector_name,proposed_weight,active_weight,total_score,momentum_score,gamma_score,option_demand_score,risk_control_score,overweight_allowed,high_vanna_risk,high_charm_risk,gex_regime_comment,vanna_charm_risk_comment
0,XLF,Financials,0.145874,2.000000e-02,1.575,2.50,1.75,0.00,0.5,True,False,True,Net GEXは正で、価格変動は抑制されやすい。現値はGamma Flipを上回る。Call...,Charm集中が高い。時間経過に伴うdealer delta調整に注意（net Charm 負）。
1,XLV,Health Care,0.109905,1.500000e-02,1.500,2.50,1.50,0.00,0.5,True,True,True,Net GEXは正で、価格変動は抑制されやすい。現値はGamma Flipを上回る。Call...,Vanna/Charmの集中が高い。IV変化と1日時間経過でdealer delta調整が出...
2,XLRE,Real Estate,0.019980,3.469447e-18,1.275,2.50,0.75,0.00,0.5,False,False,False,Net GEXは正で、価格変動は抑制されやすい。現値はGamma Flipを上回る。Call...,Vanna/Charmの集中は相対的に限定的。Gamma/GEX水準を主軸に判断。
3,XLE,Energy,0.049960,1.000000e-02,1.225,1.75,1.75,-0.25,0.5,True,False,True,Net GEXは正で、価格変動は抑制されやすい。現値はGamma Flipを上回る。Call...,Charm集中が高い。時間経過に伴うdealer delta調整に注意（net Charm 負）。
4,XLU,Utilities,0.024975,6.938894e-18,0.600,1.25,0.50,0.25,-1.0,False,False,True,Net GEXは正で、価格変動は抑制されやすい。現値はGamma Flipを上回る。Call...,Charm集中が高い。時間経過に伴うdealer delta調整に注意（net Charm 負）。
5,XLP,Consumer Staples,0.052947,1.387779e-17,0.275,1.75,-1.75,0.25,0.5,True,False,True,Net GEXは負で、ヘッジフロー由来の不安定化に注意。現値はGamma Flipを下回り、...,Charm集中が高い。時間経過に伴うdealer delta調整に注意（net Charm 負）。
6,XLC,Communication Services,0.102897,2.775558e-17,-0.525,0.25,-1.75,-0.50,0.0,False,False,False,Net GEXは負で、ヘッジフロー由来の不安定化に注意。現値はGamma Flipを下回り、...,Vanna/Charmの集中は相対的に限定的。Gamma/GEX水準を主軸に判断。
7,XLB,Materials,0.020979,3.469447e-18,-1.075,-2.00,0.25,-1.00,-1.5,False,False,False,Net GEXは正で、価格変動は抑制されやすい。現値はGamma Flipを上回る。Call...,Vanna/Charmの集中は相対的に限定的。Gamma/GEX水準を主軸に判断。
8,XLI,Industrials,0.079910,-1.000000e-02,-1.125,-0.50,-2.75,0.00,-1.0,False,False,False,Net GEXは負で、ヘッジフロー由来の不安定化に注意。現値はGamma Flipを下回り、...,Vanna/Charmの集中は相対的に限定的。Gamma/GEX水準を主軸に判断。
9,XLK,Information Technology,0.313671,-1.500000e-02,-1.325,-1.00,-2.25,0.00,-2.5,False,False,True,Net GEXは負で、ヘッジフロー由来の不安定化に注意。現値はGamma Flipを下回り、...,Charm集中が高い。時間経過に伴うdealer delta調整に注意（net Charm 負）。


,symbol,sector_name,atm_iv,expected_move_pct,expected_low,spot,expected_high,gamma_flip_inside_expected_move,call_wall_inside_expected_move,put_wall_inside_expected_move,iv_risk_score,total_score,iv_adjusted_score,iv_weekly_comment
0,XLF,Financials,0.164559,0.022789,54.836199,56.115002,57.393804,False,True,False,-0.50,1.575,1.475,7日1σ予想変動は±2.3%（±1.28）。Call Wallが予想レンジ内にあり、上値抵抗...
1,XLV,Health Care,0.184151,0.025502,156.192510,160.279999,164.367487,False,False,False,0.25,1.500,1.550,7日1σ予想変動は±2.6%（±4.09）。主要水準は予想レンジ外で、Net GEXも正のた...
2,XLRE,Real Estate,0.242805,0.033625,43.796119,45.320000,46.843880,False,True,False,-0.50,1.275,1.175,7日1σ予想変動は±3.4%（±1.52）。Call Wallが予想レンジ内にあり、上値抵抗...
3,XLE,Energy,0.288276,0.039922,55.756536,58.075001,60.393466,True,True,False,-1.50,1.225,0.925,7日1σ予想変動は±4.0%（±2.32）。Gamma Flipが予想レンジ内にあり、週内に...
4,XLU,Utilities,0.179940,0.024919,44.215051,45.345001,46.474952,True,True,True,-1.75,0.600,0.250,7日1σ予想変動は±2.5%（±1.13）。Gamma Flipが予想レンジ内にあり、週内に...
5,XLP,Consumer Staples,0.161629,0.022383,83.165857,85.070000,86.974143,False,True,False,-0.50,0.275,0.175,7日1σ予想変動は±2.2%（±1.90）。Call Wallが予想レンジ内にあり、上値抵抗...
6,XLC,Communication Services,0.215157,0.029796,107.867284,111.180000,114.492717,False,False,False,0.00,-0.525,-0.525,7日1σ予想変動は±3.0%（±3.31）。主要水準は予想レンジ外だが、Net GEXは負で...
7,XLB,Materials,0.207283,0.028706,48.724983,50.165001,51.605018,False,False,False,0.25,-1.075,-1.025,7日1σ予想変動は±2.9%（±1.44）。主要水準は予想レンジ外で、Net GEXも正のた...
8,XLI,Industrials,0.257576,0.035670,172.200349,178.570007,184.939666,False,True,False,-0.50,-1.125,-1.225,7日1σ予想変動は±3.6%（±6.37）。Call Wallが予想レンジ内にあり、上値抵抗...
9,XLK,Information Technology,0.351691,0.048704,167.618367,176.199997,184.781627,True,False,True,-1.50,-1.325,-1.625,7日1σ予想変動は±4.9%（±8.58）。Gamma Flipが予想レンジ内にあり、週内に...


,symbol,sector_name,spot,net_gex,abs_gex,gamma_flip,spot_to_flip_pct,call_wall,put_wall,spot_to_call_wall_pct,spot_to_put_wall_pct,near_5pct_gex,short_7d_gex,short_14d_gex,positive_gamma_regime,below_gamma_flip,near_gamma_flip,near_call_wall,near_put_wall,put_wall_broken,short_dated_risk,gamma_score,option_demand_score,total_score,active_weight,proposed_weight,gex_regime_comment,high_vanna_risk,high_charm_risk,vanna_abs_concentration_ratio,charm_abs_concentration_ratio,vanna_charm_risk_comment
0,XLF,Financials,56.115002,1.378159e+08,6.340553e+08,53.919449,-0.039126,56.0,50.0,-0.002049,-0.108973,1.671356e+08,1.576033e+07,2.973552e+07,True,False,False,False,False,False,False,1.75,0.00,1.575,2.000000e-02,0.145874,Net GEXは正で、価格変動は抑制されやすい。現値はGamma Flipを上回る。Call...,False,True,0.175002,0.361864,Charm集中が高い。時間経過に伴うdealer delta調整に注意（net Charm 負）。
1,XLV,Health Care,160.279999,4.565183e+07,1.888704e+08,151.744240,-0.053255,165.0,155.0,0.029448,-0.032942,2.247225e+07,6.077347e+06,4.927072e+06,True,False,False,False,False,False,False,1.50,0.00,1.500,1.500000e-02,0.109905,Net GEXは正で、価格変動は抑制されやすい。現値はGamma Flipを上回る。Call...,True,True,0.281187,0.609052,Vanna/Charmの集中が高い。IV変化と1日時間経過でdealer delta調整が出...
2,XLRE,Real Estate,45.320000,1.691641e+06,3.377738e+06,41.658543,-0.080791,46.0,35.0,0.015004,-0.227714,1.573375e+06,0.000000e+00,0.000000e+00,True,False,False,True,False,False,False,0.75,0.00,1.275,3.469447e-18,0.019980,Net GEXは正で、価格変動は抑制されやすい。現値はGamma Flipを上回る。Call...,False,False,0.233603,0.411743,Vanna/Charmの集中は相対的に限定的。Gamma/GEX水準を主軸に判断。
3,XLE,Energy,58.075001,4.433447e+07,3.722114e+08,56.349960,-0.029704,60.0,55.0,0.033147,-0.052949,7.306102e+07,1.895412e+07,3.222771e+07,True,False,False,False,False,False,False,1.75,-0.25,1.225,1.000000e-02,0.049960,Net GEXは正で、価格変動は抑制されやすい。現値はGamma Flipを上回る。Call...,False,True,0.099289,0.356579,Charm集中が高い。時間経過に伴うdealer delta調整に注意（net Charm 負）。
4,XLU,Utilities,45.345001,6.399550e+06,1.257460e+08,45.054449,-0.006408,45.0,45.0,-0.007608,-0.007608,2.001426e+07,3.211299e+05,1.083387e+07,True,False,True,False,True,False,False,0.50,0.25,0.600,6.938894e-18,0.024975,Net GEXは正で、価格変動は抑制されやすい。現値はGamma Flipを上回る。Call...,False,True,0.209586,0.637970,Charm集中が高い。時間経過に伴うdealer delta調整に注意（net Charm 負）。
5,XLP,Consumer Staples,85.070000,-2.610883e+07,1.036807e+08,88.519105,0.040544,85.0,78.0,-0.000823,-0.083108,-5.176122e+06,7.724088e+05,-1.534779e+06,False,True,False,False,False,False,False,-1.75,0.25,0.275,1.387779e-17,0.052947,Net GEXは負で、ヘッジフロー由来の不安定化に注意。現値はGamma Flipを下回り、...,False,True,0.305154,0.663978,Charm集中が高い。時間経過に伴うdealer delta調整に注意（net Charm 負）。
6,XLC,Communication Services,111.180000,-4.779397e+07,6.595149e+07,127.112991,0.143308,115.0,104.0,0.034359,-0.064580,-4.178097e+06,2.332793e+05,2.704511e+04,False,True,False,False,False,False,False,-1.75,-0.50,-0.525,2.775558e-17,0.102897,Net GEXは負で、ヘッジフロー由来の不安定化に注意。現値はGamma Flipを下回り、...,False,False,0.074257,0.154108,Vanna/Charmの集中は相対的に限定的。Gamma/GEX水準を主軸に判断。
7,XLB,Materials,50.165001,9.937728e+06,5.605245e+07,48.706030,-0.029083,55.0,47.5,0.096382,-0.053125,-2.998290e+06,-6.302927e+05,-1.164527e+06,True,False,False,False,False,False,False,0.25,-1.00,-1.075,3.469447e-18,0.020979,Net GEXは正で、価格変動は抑制されやすい。現値はGamma Flipを上回る。Call...,False,False,0.067261,0.234877,Vanna/Charmの集中は相対的に限定的。Gamma/GEX水準を主軸に判断。
8,XLI,Industrials,178.570007,-9.872173e+07,1.872596e+08,193.710579,0.084788,180.0,165.0,0.008008,-0.075993,-3.940614e+07,-4.056342e+06,-2.625241e+07,False,True,False,True,False,False,False,-2.75,0.00,-1.125,-1.000000e-02,0.079910,Net GEXは負で、ヘッジフロー由来の不安定化に注意。現値はGamma Flipを下回り、...,False,False,0.154775,0.338320,Vanna/Charmの集中は相対的に限定的。Gamma/GEX水準を主軸に判断。
9,XLK,Information Technology,176.199997,-1.399264e+07,1.436385e+08,180.693901,0.025505,200.0,170.0,0.135074,-0.035187,-1.680857e+07,-4.280914e+06,-6.330462e+06,False,True,False,False,False,False,False,-2.25,0.00,-1.325,-1.500000e-02,0.313671,Net GEXは負で、ヘッジフロー由来の不安定化に注意。現値はGamma Flipを下回り、...,False,True,0.110774,0.457760,Charm集中が高い。時間経過に伴うdeal

## Dashboard 実行


In [18]:
dashboard_figures = {}
if not proposed_portfolio.empty:
    dashboard_figures["allocation"] = plot_allocation(proposed_portfolio)
    dashboard_figures["active_tilt"] = plot_active_weights(proposed_portfolio)
    dashboard_figures["signal_heatmap"] = plot_signal_heatmap(proposed_portfolio)
    dashboard_figures["gex_indicator_heatmap"] = plot_gex_indicator_heatmap(proposed_portfolio)
    dashboard_figures["gex_by_strike"] = plot_gex_by_strike(option_chains, proposed_portfolio)
    dashboard_figures["gamma_flip_curve"] = plot_gamma_flip_curve(gex_grids, proposed_portfolio)
    dashboard_figures["price_with_walls"] = plot_price_with_walls(price_df, proposed_portfolio)
    dashboard_figures["iv_expected_move_bands"] = plot_iv_expected_move_bands(iv_expected_moves)
    dashboard_figures["iv_overlay_heatmap"] = plot_iv_overlay_heatmap(iv_expected_moves)
    dashboard_figures["Vanna/Charm Exposure Heatmap"] = plot_vanna_charm_exposure_heatmap(second_order_greeks)
    dashboard_figures["Vanna/Charm by Strike"] = plot_vanna_charm_by_strike(option_chains, proposed_portfolio)

for name, fig in dashboard_figures.items():
    print(name)
    fig.show()

display(HTML("<h3>GEX raw values</h3>"))
display(gex_summary)
display(HTML("<h3>Vanna/Charm Second-Order Greeks</h3>"))
display(second_order_greeks)
display(HTML("<h3>IV Expected Move Overlay</h3>"))
display(iv_expected_moves)


allocation


active_tilt


signal_heatmap


gex_indicator_heatmap


gex_by_strike


...

Vanna/Charm Exposure Heatmap


Vanna/Charm by Strike


,symbol,spot,net_gex,abs_gex,gamma_flip,spot_to_flip_pct,call_wall,put_wall,spot_to_call_wall_pct,spot_to_put_wall_pct,near_2pct_gex,near_2pct_abs_gex,near_5pct_gex,near_5pct_abs_gex,short_7d_gex,short_7d_abs_gex,short_14d_gex,short_14d_abs_gex,near_5pct_abs_gex_ratio,short_7d_abs_gex_ratio,short_14d_abs_gex_ratio,call_oi,put_oi,call_volume,put_volume,put_call_oi_ratio,put_call_volume_ratio,atm_iv
0,XLK,176.199997,-1.399264e+07,1.436385e+08,180.693901,0.025505,200.0,170.0,0.135074,-0.035187,-1.059216e+07,2.294229e+07,-1.680857e+07,6.309551e+07,-4.280914e+06,2.404355e+07,-6.330462e+06,3.376645e+07,0.439266,0.167389,0.235079,272163.0,424216.0,12025.0,38066.0,1.558684,3.165572,0.351691
1,XLY,114.849998,-7.349819e+07,1.291945e+08,137.819998,0.200000,120.0,107.5,0.044841,-0.063997,-1.456992e+07,2.195361e+07,-3.768590e+07,6.328257e+07,1.873483e+05,1.090196e+06,1.641009e+06,3.183539e+06,0.489824,0.008438,0.024641,112890.0,416615.0,9761.0,136134.0,3.690451,13.946727,0.255501
2,XLP,85.070000,-2.610883e+07,1.036807e+08,88.519105,0.040544,85.0,78.0,-0.000823,-0.083108,-1.086072e+06,4.707337e+07,-5.176122e+06,7.054191e+07,7.724088e+05,1.407201e+07,-1.534779e+06,2.251874e+07,0.680376,0.135724,0.217193,82097.0,264345.0,6721.0,14147.0,3.219911,2.104895,0.161629
3,XLE,58.075001,4.433447e+07,3.722114e+08,56.349960,-0.029704,60.0,55.0,0.033147,-0.052949,2.880297e+07,8.405278e+07,7.306102e+07,1.577905e+08,1.895412e+07,2.514700e+07,3.222771e+07,4.054917e+07,0.423927,0.067561,0.108941,1761194.0,2552053.0,30665.0,66677.0,1.449047,2.174368,0.288276
4,XLF,56.115002,1.378159e+08,6.340553e+08,53.919449,-0.039126,56.0,50.0,-0.002049,-0.108973,1.234163e+08,2.125486e+08,1.671356e+08,3.176333e+08,1.576033e+07,3.118879e+07,2.973552e+07,6.063562e+07,0.500955,0.049189,0.095631,2320538.0,3129442.0,22993.0,91919.0,1.348585,3.997695,0.164559
5,XLV,160.279999,4.565183e+07,1.888704e+08,151.744240,-0.053255,165.0,155.0,0.029448,-0.032942,2.723282e+06,5.410207e+07,2.247225e+07,1.154142e+08,6.077347e+06,2.059804e+07,4.927072e+06,3.013176e+07,0.611076,0.109059,0.159537,228687.0,315320.0,8735.0,10471.0,1.378828,1.198741,0.184151
6,XLI,178.570007,-9.872173e+07,1.872596e+08,193.710579,0.084788,180.0,165.0,0.008008,-0.075993,-2.698438e+07,5.332361e+07,-3.940614e+07,8.798294e+07,-4.056342e+06,9.086092e+06,-2.625241e+07,3.572460e+07,0.469845,0.048521,0.190776,100407.0,341529.0,8002.0,86378.0,3.401446,10.794551,0.257576
7,XLC,111.180000,-4.779397e+07,6.595149e+07,127.112991,0.143308,115.0,104.0,0.034359,-0.064580,-2.435956e+06,7.263756e+06,-4.178097e+06,1.438559e+07,2.332793e+05,1.232505e+06,2.704511e+04,2.112573e+06,0.218124,0.018688,0.032032,29373.0,211932.0,2807.0,4384.0,7.215198,1.561810,0.215157
8,XLB,50.165001,9.937728e+06,5.605245e+07,48.706030,-0.029083,55.0,47.5,0.096382,-0.053125,-1.937379e+06,9.477410e+06,-2.998290e+06,1.516829e+07,-6.302927e+05,4.231539e+06,-1.164527e+06,6.677665e+06,0.270609,0.075492,0.119132,302363.0,191161.0,5180.0,125223.0,0.632224,24.174324,0.207283
9,XLRE,45.320000,1.691641e+06,3.377738e+06,41.658543,-0.080791,46.0,35.0,0.015004,-0.227714,1.365497e+06,1.646564e+06,1.573375e+06,2.174781e+06,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.643857,0.000000,0.000000,14782.0,13906.0,327.0,1943.0,0.940739,5.941896,0.242805


,symbol,spot,net_vanna_exposure,abs_vanna_exposure,net_charm_exposure,abs_charm_exposure,near_5pct_vanna_exposure,near_5pct_charm_exposure,short_7d_vanna_exposure,short_7d_charm_exposure,near_5pct_abs_vanna_exposure,near_5pct_abs_charm_exposure,short_7d_abs_vanna_exposure,short_7d_abs_charm_exposure,vanna_near_5pct_abs_ratio,vanna_short_7d_abs_ratio,charm_near_5pct_abs_ratio,charm_short_7d_abs_ratio,vanna_abs_concentration_ratio,charm_abs_concentration_ratio,high_vanna_risk,high_charm_risk,vanna_charm_risk_comment
0,XLK,176.199997,2.132852e+07,3.519118e+07,-1.104138e+07,1.929094e+07,1.413941e+06,-3.674844e+06,8.876421e+05,-5.048284e+06,3.898277e+06,7.858421e+06,1.507167e+06,8.830629e+06,0.110774,0.042828,0.407363,0.457760,0.110774,0.457760,False,True,Charm集中が高い。時間経過に伴うdealer delta調整に注意（net Charm 負）。
1,XLY,114.849998,2.592458e+07,3.095357e+07,-4.318104e+06,5.659126e+06,5.217037e+06,-1.076153e+06,6.038809e+04,-3.192556e+05,7.426447e+06,1.945678e+06,7.483670e+04,3.949949e+05,0.239922,0.002418,0.343812,0.069798,0.239922,0.343812,False,False,Vanna/Charmの集中は相対的に限定的。Gamma/GEX水準を主軸に判断。
2,XLP,85.070000,2.071558e+07,2.429952e+07,-3.083413e+06,5.777267e+06,4.761342e+06,-1.237341e+06,7.829246e+04,-9.657430e+04,7.415095e+06,3.835980e+06,7.717312e+05,2.097072e+06,0.305154,0.031759,0.663978,0.362987,0.305154,0.663978,False,True,Charm集中が高い。時間経過に伴うdealer delta調整に注意（net Charm 負）。
3,XLE,58.075001,7.218050e+07,1.094850e+08,-1.331902e+07,2.503195e+07,4.311975e+06,-2.819680e+06,4.335660e+05,-1.796887e+06,1.087064e+07,8.925868e+06,1.298020e+06,5.898182e+06,0.099289,0.011856,0.356579,0.235626,0.099289,0.356579,False,True,Charm集中が高い。時間経過に伴うdealer delta調整に注意（net Charm 負）。
4,XLF,56.115002,1.325814e+08,2.068965e+08,-1.910081e+07,3.437624e+07,1.974851e+06,-4.575177e+06,1.415676e+06,-3.358517e+06,3.620736e+07,1.243953e+07,2.737988e+06,6.910232e+06,0.175002,0.013234,0.361864,0.201018,0.175002,0.361864,False,True,Charm集中が高い。時間経過に伴うdealer delta調整に注意（net Charm 負）。
5,XLV,160.279999,2.967028e+07,4.891772e+07,-7.819291e+06,1.137850e+07,6.283537e+06,-4.482307e+06,1.142777e+06,-2.755271e+06,1.375504e+07,6.930099e+06,1.603592e+06,4.462570e+06,0.281187,0.032781,0.609052,0.392193,0.281187,0.609052,True,True,Vanna/Charmの集中が高い。IV変化と1日時間経過でdealer delta調整が出...
6,XLI,178.570007,4.423134e+07,4.862018e+07,-1.055928e+07,1.313370e+07,4.864004e+06,-2.171818e+06,3.945274e+05,-1.492164e+06,7.525189e+06,4.443392e+06,6.259834e+05,2.420405e+06,0.154775,0.012875,0.338320,0.184290,0.154775,0.338320,False,False,Vanna/Charmの集中は相対的に限定的。Gamma/GEX水準を主軸に判断。
7,XLC,111.180000,1.971091e+07,2.134616e+07,-3.290343e+06,3.838832e+06,7.267156e+05,-1.896093e+05,3.363218e+04,-1.659075e+05,1.585107e+06,5.915956e+05,6.270641e+04,3.184843e+05,0.074257,0.002938,0.154108,0.082964,0.074257,0.154108,False,False,Vanna/Charmの集中は相対的に限定的。Gamma/GEX水準を主軸に判断。
8,XLB,50.165001,1.666799e+07,1.773501e+07,-4.340795e+06,4.935144e+06,6.549629e+05,-6.375190e+05,1.342727e+05,-5.404525e+05,1.192871e+06,1.159151e+06,2.299326e+05,8.496847e+05,0.067261,0.012965,0.234877,0.172170,0.067261,0.234877,False,False,Vanna/Charmの集中は相対的に限定的。Gamma/GEX水準を主軸に判断。
9,XLRE,45.320000,6.301223e+05,8.520552e+05,-9.990664e+04,1.314335e+05,1.014969e+05,-4.744527e+04,0.000000e+00,0.000000e+00,1.990430e+05,5.411680e+04,0.000000e+00,0.000000e+00,0.233603,0.000000,0.411743,0.000000,0.233603,0.411743,False,False,Vanna/Charmの集中は相対的に限定的。Gamma/GEX水準を主軸に判断。


,symbol,sector_name,spot,atm_iv,expected_move_days,expected_move_pct,expected_move_abs,expected_low,expected_high,gamma_flip,call_wall,put_wall,gamma_flip_inside_expected_move,call_wall_inside_expected_move,put_wall_inside_expected_move,expected_move_to_gamma_flip_ratio,expected_move_to_call_wall_ratio,expected_move_to_put_wall_ratio,iv_risk_score,total_score,iv_adjusted_score,proposed_weight,active_weight,iv_weekly_comment
0,XLF,Financials,56.115002,0.164559,7,0.022789,1.278803,54.836199,57.393804,53.919449,56.0,50.0,False,True,False,0.582451,11.119862,0.209125,-0.50,1.575,1.475,0.145874,2.000000e-02,7日1σ予想変動は±2.3%（±1.28）。Call Wallが予想レンジ内にあり、上値抵抗...
1,XLV,Health Care,160.279999,0.184151,7,0.025502,4.087488,156.192510,164.367487,151.744240,165.0,155.0,False,False,False,0.478866,0.865993,0.774146,0.25,1.500,1.550,0.109905,1.500000e-02,7日1σ予想変動は±2.6%（±4.09）。主要水準は予想レンジ外で、Net GEXも正のた...
2,XLRE,Real Estate,45.320000,0.242805,7,0.033625,1.523880,43.796119,46.843880,41.658543,46.0,35.0,False,True,False,0.416195,2.241000,0.147663,-0.50,1.275,1.175,0.019980,3.469447e-18,7日1σ予想変動は±3.4%（±1.52）。Call Wallが予想レンジ内にあり、上値抵抗...
3,XLE,Energy,58.075001,0.288276,7,0.039922,2.318465,55.756536,60.393466,56.349960,60.0,55.0,True,True,False,1.344006,1.204398,0.753972,-1.50,1.225,0.925,0.049960,1.000000e-02,7日1σ予想変動は±4.0%（±2.32）。Gamma Flipが予想レンジ内にあり、週内に...
4,XLU,Utilities,45.345001,0.179940,7,0.024919,1.129950,44.215051,46.474952,45.054449,45.0,45.0,True,True,True,3.888981,3.275207,3.275207,-1.75,0.600,0.250,0.024975,6.938894e-18,7日1σ予想変動は±2.5%（±1.13）。Gamma Flipが予想レンジ内にあり、週内に...
5,XLP,Consumer Staples,85.070000,0.161629,7,0.022383,1.904143,83.165857,86.974143,88.519105,85.0,78.0,False,True,False,0.552069,27.202163,0.269327,-0.50,0.275,0.175,0.052947,1.387779e-17,7日1σ予想変動は±2.2%（±1.90）。Call Wallが予想レンジ内にあり、上値抵抗...
6,XLC,Communication Services,111.180000,0.215157,7,0.029796,3.312717,107.867284,114.492717,127.112991,115.0,104.0,False,False,False,0.207916,0.867203,0.461381,0.00,-0.525,-0.525,0.102897,2.775558e-17,7日1σ予想変動は±3.0%（±3.31）。主要水準は予想レンジ外だが、Net GEXは負で...
7,XLB,Materials,50.165001,0.207283,7,0.028706,1.440017,48.724983,51.605018,48.706030,55.0,47.5,False,False,False,0.987009,0.297832,0.540344,0.25,-1.075,-1.025,0.020979,3.469447e-18,7日1σ予想変動は±2.9%（±1.44）。主要水準は予想レンジ外で、Net GEXも正のた...
8,XLI,Industrials,178.570007,0.257576,7,0.035670,6.369658,172.200349,184.939666,193.710579,180.0,165.0,False,True,False,0.420701,4.454329,0.469392,-0.50,-1.125,-1.225,0.079910,-1.000000e-02,7日1σ予想変動は±3.6%（±6.37）。Call Wallが予想レンジ内にあり、上値抵抗...
9,XLK,Information Technology,176.199997,0.351691,7,0.048704,8.581630,167.618367,184.781627,180.693901,200.0,170.0,True,False,True,1.909616,0.360573,1.384135,-1.50,-1.325,-1.625,0.313671,-1.500000e-02,7日1σ予想変動は±4.9%（±8.58）。Gamma Flipが予想レンジ内にあり、週内に...


## Save Outputs 実行


In [19]:
output_paths = save_outputs(
    chains_by_symbol=option_chains,
    gex_summary_df=gex_summary,
    sector_scores_df=sector_scores,
    proposed_portfolio_df=proposed_portfolio,
    gex_decision_table_df=gex_decision_table,
    figures=dashboard_figures,
    iv_expected_moves_df=iv_expected_moves,
    second_order_greeks_df=second_order_greeks,
    weekly_attribution_df=weekly_attribution,
    weekly_attribution_summary=weekly_attribution_summary,
    as_of=AS_OF,
)

display(pd.DataFrame({"artifact": list(output_paths.keys()), "path": list(output_paths.values())}))


,artifact,path
0,raw_options_XLK,/Users/kencharoff/workspace/projects/sector-ga...
1,raw_options_XLY,/Users/kencharoff/workspace/projects/sector-ga...
2,raw_options_XLP,/Users/kencharoff/workspace/projects/sector-ga...
3,raw_options_XLE,/Users/kencharoff/workspace/projects/sector-ga...
4,raw_options_XLF,/Users/kencharoff/workspace/projects/sector-ga...
5,raw_options_XLV,/Users/kencharoff/workspace/projects/sector-ga...
6,raw_options_XLI,/Users/kencharoff/workspace/projects/sector-ga...
7,raw_options_XLC,/Users/kencharoff/workspace/projects/sector-ga...
8,raw_options_XLB,/Users/kencharoff/workspace/projects/sector-ga...
9,raw_options_XLRE,/Users/kencharoff/workspace/projects/sector-ga...


## Validation 実行


In [20]:
validation_results = validate_outputs(
    chains_by_symbol=option_chains,
    gex_summary_df=gex_summary,
    proposed_portfolio_df=proposed_portfolio,
    figures=dashboard_figures,
    output_paths=output_paths,
    iv_expected_moves_df=iv_expected_moves,
    second_order_greeks_df=second_order_greeks,
    weekly_attribution_df=weekly_attribution,
    weekly_attribution_summary=weekly_attribution_summary,
)

display(validation_results)


,category,check,status,detail
0,data,XLK option chain non-empty,PASS,"rows=2,180"
1,data,XLK openInterest sum non-zero,PASS,696379.0
2,data,XLK impliedVolatility mostly valid,PASS,valid_ratio=100.0%
3,data,XLK gamma not all NaN,PASS,
4,data,XLK vanna_1vol not all NaN,PASS,
...,...,...,...,...
113,iv,iv_expected_move_table csv saved,PASS,/Users/kencharoff/workspace/projects/sector-ga...
114,second_order,sector_second_order_greeks parquet saved,PASS,/Users/kencharoff/workspace/projects/sector-ga...
115,second_order,second_order_greeks_table csv saved,PASS,/Users/kencharoff/workspace/projects/sector-ga...
116,attribution,weekly attribution parquet saved,PASS,/Users/kencharoff/workspace/projects/sector-ga...


## 12. Limitations

- yFinance から取得できるオプションチェーンは現在時点のスナップショットであり、過去チェーンを完全に再現するものではない。
- Signed GEX は call を正、put を負とする簡易プロキシであり、実際のディーラーのポジション方向を直接推定していない。
- Open Interest、Volume、IV は yFinance 側の更新タイミングや欠損の影響を受ける。
- IV Expected Move は `atm_iv * sqrt(7 / 365)` による1σ相当の週次予想変動であり、方向予測ではない。
- 前週パフォーマンス検証は週初直前と週末の配当調整済み終値によるポイント・ツー・ポイント評価であり、売買コスト、税、週中リバランスは含まない。
- `momentum_score` は指示書で明示式がないため、20日/60日絶対リターン、SPY対比相対リターン、20MA/50MA、60日ドローダウンを組み合わせた実装上のルールとして定義している。
- 実運用や厳密なバックテストでは、ThetaData、ORATS、Cboe、OptionMetrics などの正式データソースへの差し替えを検討する。
